# Notebook introduction

This notebook runs a Monte Carlo feasibility analysis for quantum repeater links.

### How to run the next cell
- Select the next code cell and press Shift+Enter (or click ▶) to initialize imports, configuration, and global state.
- If execution fails due to missing packages, install those listed in `requirements.txt` and re-run.
- Re-run this initialization cell after changing configuration files in `config/`.

(Scroll for detailed mathematical and physical background.)

In [ ]:
# Utility: Convert CONFIG/WORKING_MARGINALS into locked WORKING_MARGINALS_LIST and VAR_ORDER and build_copula(var_order, kendall_tau_pairs) per final spec.

import numpy as _np
from math import sin, pi
from scipy.interpolate import interp1d

CANONICAL_VARS = [
    'alpha_dB_per_km','eta_s','eta_w','eta_r','eta_det','V0','T2_ms',
    'p_dark','raman_rate_per_ns','alpha_dB_per_km', 'L0_km','N_modes','n_swaps','kappa_power'
]
# c_fiber_mps is a constant and must not appear in WORKING_MARGINALS
CONSTANT_KEYS = ['c_fiber_mps']


def convert_to_working_marginals(defaults=None, working=None):
    """Take DEFAULT_MARGINALS (dict) and WORKING_MARGINALS (dict) and produce
    WORKING_MARGINALS_LIST (ordered list of marginal dicts with ppf/cdf callables)
    and VAR_ORDER. Also ensure CONFIG['constants']['c_fiber_mps'] exists.
    """
    defaults = defaults or globals().get('DEFAULT_MARGINALS', {})
    working = working or globals().get('WORKING_MARGINALS', {})
    cfg = globals().get('CONFIG') or {}
    cfg.setdefault('constants', {})

    # Ensure c_fiber_mps is placed into CONFIG.constants (from working or defaults)
    if 'c_fiber_mps' in working:
        try:
            v = float(working['c_fiber_mps'].get('params',{}).get('value'))
            cfg['constants']['c_fiber_mps'] = v
        except Exception:
            pass
    elif 'c_fiber_mps' in defaults:
        try:
            v = float(defaults['c_fiber_mps'].get('params',{}).get('value'))
            cfg['constants']['c_fiber_mps'] = v
        except Exception:
            pass
    globals()['CONFIG'] = cfg

    # Build marginals in canonical order: prefer working over defaults
    marg_list = []
    # determine order: use Canonical if present otherwise keys from working
    order = [k for k in [
        'alpha_dB_per_km','eta_s','eta_w','eta_r','eta_det','V0','T2_ms','p_dark','raman_rate_per_ns','L0_km','N_modes','n_swaps','kappa_power'
    ] if (k in working or k in defaults)]

    def _bind(spec):
        # return (family_lower, params, unit, ppf, cdf)
        from scipy.stats import beta as _beta, lognorm as _lognorm, truncnorm as _truncnorm, norm as _norm, uniform as _uniform
        if not isinstance(spec, dict):
            spec = {'dist': 'normal', 'params': {'mu': float(spec)}}
        fam = (spec.get('family') or spec.get('dist') or '').lower()
        params = spec.get('params', {}) or {}
        unit = spec.get('unit','')
        if fam in ('beta','beta_dist'):
            a=float(params.get('a',1.0)); b=float(params.get('b',1.0));
            return 'beta', params, unit, (lambda u, a=a,b=b: _beta.ppf(u,a,b)), (lambda x, a=a,b=b: _beta.cdf(x,a,b))
        if fam in ('lognormal','log-normal','lognorm'):
            mu=float(params.get('mu_log', params.get('mu',0.0))); sigma=float(params.get('sigma_log', params.get('sigma',1.0)))
            s=sigma; scale=_np.exp(mu)
            return 'lognormal', params, unit, (lambda u, s=s,scale=scale: _lognorm.ppf(u,s,scale=scale)), (lambda x, s=s,scale=scale: _lognorm.cdf(x,s,scale=scale))
        if fam in ('normal','gaussian'):
            mu=float(params.get('mu',0.0)); sigma=max(1e-12,float(params.get('sigma',1.0)))
            return 'normal', params, unit, (lambda u, mu=mu,sigma=sigma: _norm.ppf(u,loc=mu,scale=sigma)), (lambda x, mu=mu,sigma=sigma: _norm.cdf(x,loc=mu,scale=sigma))
        if fam in ('uniform',):
            low=float(params.get('low',0.0)); high=float(params.get('high',1.0))
            if high==low: high=low+1e-12
            return 'uniform', params, unit, (lambda u, low=low,high=high: _uniform.ppf(u,loc=low,scale=high-low)), (lambda x, low=low,high=high: _uniform.cdf(x,loc=low,scale=high-low))
        if fam in ('truncnorm','truncated_normal'):
            lo=float(params.get('low',0.0)); hi=float(params.get('high',1.0)); loc=float(params.get('loc',0.0)); scale=float(params.get('scale',1.0))
            a=(lo-loc)/scale; b=(hi-loc)/scale
            return 'truncnorm', params, unit, (lambda u, a=a,b=b,loc=loc,scale=scale: _truncnorm.ppf(u,a,b,loc=loc,scale=scale)), (lambda x, a=a,b=b,loc=loc,scale=scale: _truncnorm.cdf(x,a,b,loc=loc,scale=scale))
        if fam in ('empirical','ecdf'):
            # Expect params['samples'] to be an array-like or params['values'] with weights
            samples = _np.asarray(params.get('samples') or params.get('values') or [])
            if samples.size==0:
                # fallback to constant zero
                return 'empirical', params, unit, (lambda u: _np.zeros_like(u,dtype=float)), (lambda x: _np.zeros_like(x,dtype=float))
            # build ecdf and ppf via interpolation on sorted samples
            xs = _np.sort(samples)
            ps = _np.linspace(0,1,len(xs),endpoint=False) + (0.5/len(xs))
            ppf_fn = interp1d(ps, xs, bounds_error=False, fill_value=(xs[0], xs[-1]))
            def ppf(u):
                uarr = _np.asarray(u)
                uclip = _np.clip(uarr, ps[0], ps[-1])
                return ppf_fn(uclip)
            def cdf(x):
                xarr = _np.asarray(x)
                return _np.searchsorted(xs, xarr, side='right') / float(len(xs))
            return 'empirical', params, unit, ppf, cdf
        # const
        if fam in ('const','constant') or fam=='':
            val=float(params.get('value',0.0))
            return 'const', params, unit, (lambda u, v=val: _np.full_like(_np.asarray(u,dtype=float), v, dtype=float)), (lambda x, v=val: _np.where(_np.asarray(x)>=v,1.0,0.0))
        # unknown
        raise ValueError(f'Unsupported family: {fam}')

    for key in order:
        if key in CONSTANT_KEYS:
            continue
        spec = working.get(key) or defaults.get(key)
        if not spec:
            # skip missing optional
            continue
        try:
            family, params, unit, ppf, cdf = _bind(spec)
        except Exception as e:
            raise
        marg_list.append({'key': key, 'unit': unit, 'family': family, 'params': dict(params), 'ppf': ppf, 'cdf': cdf})

    globals()['WORKING_MARGINALS_LIST'] = marg_list
    globals()['VAR_ORDER'] = [m['key'] for m in marg_list]
    FLOW_STATE['WORKING_MARGINALS_LIST'] = marg_list
    FLOW_STATE['VAR_ORDER'] = globals()['VAR_ORDER']
    print('[convert] Created WORKING_MARGINALS_LIST with', len(marg_list), 'entries')
    return marg_list


def build_copula(var_order, kendall_tau_pairs):
    """Build Gaussian copula L and Sigma_psd from kendall tau pairs per spec.
    Returns dict {Sigma_psd, L, diag:{min_eig,max_eig,psd_correction_norm}}
    """
    d = len(var_order)
    idx = {k:i for i,k in enumerate(var_order)}
    tau_mat = _np.zeros((d,d))
    for a,b,t in kendall_tau_pairs:
        if a in idx and b in idx:
            ia,ib = idx[a], idx[b]
            tau_mat[ia,ib] = tau_mat[ib,ia] = float(t)
    # Convert to Pearson rho
    rho = _np.sin(_np.pi * tau_mat / 2.0)
    _np.fill_diagonal(rho, 1.0)
    rho = (rho + rho.T) / 2.0
    # PSD repair
    vals, vecs = _np.linalg.eigh(rho)
    clipped = _np.clip(vals, 1e-12, None)
    psd = vecs @ _np.diag(clipped) @ vecs.T
    # Renormalize to correlation
    dvec = _np.sqrt(_np.clip(_np.diag(psd), 1e-12, None))
    Sigma_psd = psd / _np.outer(dvec, dvec)
    Sigma_psd = (Sigma_psd + Sigma_psd.T)/2.0
    frob = float(_np.linalg.norm(rho - Sigma_psd))
    eigvals = _np.linalg.eigvalsh(Sigma_psd)
    try:
        L = _np.linalg.cholesky(Sigma_psd + 1e-12 * _np.eye(d))
    except Exception:
        L = _np.eye(d)
    return {'Sigma_psd': Sigma_psd, 'L': L, 'diag': {'min_eig': float(eigvals.min()), 'max_eig': float(eigvals.max()), 'psd_correction_norm': frob}}

print('copula-builder utilities ready')


# Mathematical overview and code map

This notebook performs Monte Carlo feasibility analysis for quantum repeater links using:
1. Marginal distributions $F_j$ with quantile functions $F_j^{-1}$.
2. A Gaussian copula for dependence (Sklar 1959; Nelsen 2006 https://doi.org/10.1007/978-0-387-28678-0; PSD repair via Higham 2002 https://doi.org/10.1093/imanum/22.3.329).
3. Physics kernels for link metrics and throughput $R$ (Duan et al. 2001 https://doi.org/10.1038/35106500; Briegel et al. 1998 https://doi.org/10.1103/PhysRevLett.81.5932).
4. Diagnostics: Probability Integral Transform (Dawid 1984 https://doi.org/10.2307/2982829), Kolmogorov–Smirnov test (Massey 1951 https://doi.org/10.1080/01621459.1951.10500769), Kendall’s $\tau$ (Kendall 1938 https://doi.org/10.2307/2332226), KDE (Silverman 1986 https://doi.org/10.1007/978-1-4899-3324-9).

### How to run the next cell
- Run the next code cell with Shift+Enter (or click ▶) to set up configuration paths and any early-state checks.
- If you modify preset selection or parameters above, re-run this cell to propagate changes.

---
## Quantum mechanics and linear algebra primer (Dirac notation)
States $|\psi\rangle$, measurement projectors $\Pi_i$, Born rule $p_i=\langle\psi|\Pi_i|\psi\rangle$, density matrix $\rho$. Entanglement: Bell states (e.g. $|\Phi^+\rangle=(|00\rangle+|11\rangle)/\sqrt{2}$). Observables $A=\vec a\cdot\vec \sigma$ for qubits. Reduced states via partial trace encode local statistics—crucial for distinguishing classical vs quantum correlations.

### Classical vs quantum correlations (stepping toward Bell)
- Classical (local hidden variable) model: outcomes $A(a,\lambda), B(b,\lambda)$ with shared parameter $\lambda$.
- Quantum: expectation $E(a,b)=\langle\psi| (\vec a\cdot\vec\sigma)\otimes(\vec b\cdot\vec\sigma) |\psi\rangle$ can exceed classical bounds (Bell violation) for entangled $|\psi\rangle$.

This distinction motivates Bell/CHSH tests used in E91 and device-independent approaches (see dedicated Bell section).

---
## Raman scattering and Ramsey measurement
Raman write/read create and retrieve spin waves via off-resonant two-photon processes (Fleischhauer et al. 2005 https://doi.org/10.1103/RevModPhys.77.633); background within detection window sets noise rate $r_{\text{raman}}$. Ramsey fringes $V(\tau)\approx e^{-\tau/T_2}$ motivate memory survival factor $s=e^{-\tau_{\text{oneway}}/T_2}$ (Ramsey 1950 https://doi.org/10.1103/PhysRev.78.695).

---
## Evolution of QKD protocols
| Era | Protocol | Key Idea | Security Feature | Reference |
|-----|----------|----------|------------------|-----------|
| 1984 | BB84 | Two bases (Z/X) | Basis mismatch reveals eavesdropping | Bennett & Brassard 1984 reprint DOI: https://doi.org/10.1016/j.tcs.2014.05.025 |
| Early 90s | B92 | Two non-orthogonal states | Unambiguous state discrimination limits Eve | Bennett 1992 DOI: https://doi.org/10.1103/PhysRevLett.68.3121 |
| 1991 | E91 (Ekert) | Entanglement + Bell test | CHSH violation bounds Eve’s info | Ekert. A, Phys. Rev. Lett. 67, 661-663 (1991) (doi link not found) |
| Late 90s | Six-state | Three mutually unbiased bases | Higher symmetry improves error estimation | Bruss 1998 DOI: https://doi.org/10.1103/PhysRevLett.81.3018 |
| 2003–05 | Decoy-state BB84 | Vary mean photon number | Detect photon-number-splitting attacks | Lo, Ma, Chen 2005 DOI: https://doi.org/10.1103/PhysRevLett.94.230504 |
| 2012 | MDI-QKD | Untrusted measurement station | Removes detector side-channel | Lo, Curty, Qi 2012 DOI: https://doi.org/10.1103/PhysRevLett.108.130503 |
| 2010+ | DI-QKD | Device-independence via Bell | Security from observed nonlocality only | Acín et al. 2007 DOI: https://doi.org/10.1103/PhysRevLett.98.230501 |

(See the dedicated Bell/CHSH section below for mathematical details.)


In [1]:
# Auto-install missing dependencies (ipywidgets, plotly, scipy) if absent
import sys, subprocess, importlib, json, time
needed = ['ipywidgets','plotly','scipy']
installed = {}
for pkg in needed:
    try:
        importlib.import_module(pkg)
        installed[pkg] = 'present'
    except Exception:
        print(f'Installing {pkg} ...')
        code = subprocess.call([sys.executable,'-m','pip','install',pkg])
        installed[pkg] = f'installed code={code}'
print('Dependency status:', installed)
req_path = 'requirements.txt'
with open(req_path,'w',encoding='utf-8') as f:
    for pkg in needed:
        f.write(pkg+'\n')
print('Wrote', req_path)

Dependency status: {'ipywidgets': 'present', 'plotly': 'present', 'scipy': 'present'}
Wrote requirements.txt


In [2]:
# 2) Global Constants, Paths, and Reproducibility Stamp
import numpy as np
from pathlib import Path
import hashlib
import json as _json

GLOBAL_SEED = 123456789
np.random.seed(GLOBAL_SEED)
RNG = np.random.default_rng(GLOBAL_SEED)

ROOT = Path('.')
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
CONFIG_DIR = ROOT / 'config'
CONFIG_DIR.mkdir(exist_ok=True)
(CONFIG_DIR / 'presets').mkdir(parents=True, exist_ok=True)

SCHEMA = {
    'alpha_dB_per_km':    {'type': float, 'unit': 'dB/km', 'min': 0.0, 'default': 0.2},
    'L0_km':              {'type': float, 'unit': 'km',    'min': 0.0, 'default': 25.0},
    'eta_s':              {'type': float, 'unit': 'frac',  'min': 0.0, 'max': 1.0, 'default': 0.6},
    'eta_w':              {'type': float, 'unit': 'frac',  'min': 0.0, 'max': 1.0, 'default': 0.7},
    'eta_r':              {'type': float, 'unit': 'frac',  'min': 0.0, 'max': 1.0, 'default': 0.7},
    'eta_det':            {'type': float, 'unit': 'frac',  'min': 0.0, 'max': 1.0, 'default': 0.9},
    'V0':                 {'type': float, 'unit': 'vis',   'min': 0.0, 'max': 1.0, 'default': 0.9},
    'T2_ms':              {'type': float, 'unit': 'ms',    'min': 0.01, 'default': 150.0},
    'N_modes':            {'type': float, 'unit': 'count', 'min': 1.0, 'default': 50.0},
    'n_swaps':            {'type': float, 'unit': 'count', 'min': 0.0, 'default': 3.0},
    'kappa_power':        {'type': float, 'unit': 'power', 'min': 0.0, 'default': 1.0},
    'c_fiber_mps':        {'type': float, 'unit': 'm/s',   'min': 1e6, 'default': 2.0e8},
    'geometry':           {'type': str,   'unit': 'enum',  'choices': ['end_detection','central_bsm'], 'default': 'central_bsm'},
    'kappa_mode':         {'type': str,   'unit': 'enum',  'choices': ['power','linear','square'], 'default': 'power'},
    'connector_loss_dB':  {'type': float, 'unit': 'dB',    'min': 0.0, 'default': 0.5},
    'n_connectors':       {'type': int,   'unit': 'count', 'min': 0,   'default': 2},
    'splice_loss_dB':     {'type': float, 'unit': 'dB',    'min': 0.0, 'default': 0.1},
    'n_splices':          {'type': int,   'unit': 'count', 'min': 0,   'default': 0},
    'filter_loss_dB':     {'type': float, 'unit': 'dB',    'min': 0.0, 'default': 0.0},
    'p_dark':             {'type': float, 'unit': 'prob',  'min': 0.0, 'max': 1.0, 'default': 0.0},
    'raman_rate_per_ns':  {'type': float, 'unit': 'prob/ns','min': 0.0, 'default': 0.0},
    'gate_ns':            {'type': float, 'unit': 'ns',    'min': 0.0, 'default': 1.0},
}

def _schema_for_hash():
    def clean(v):
        if isinstance(v, dict):
            return {k: clean(vv) for k, vv in v.items()}
        if isinstance(v, (list, tuple)):
            return [clean(x) for x in v]
        if isinstance(v, type):
            return v.__name__
        return v
    return clean(SCHEMA)

def reproducibility_stamp(extra=None):
    h = hashlib.sha256()
    h.update(str(GLOBAL_SEED).encode())
    serializable_schema = _schema_for_hash()
    h.update(_json.dumps(serializable_schema, sort_keys=True).encode())
    if extra:
        h.update(_json.dumps(extra, sort_keys=True).encode())
    return h.hexdigest()[:16]

print('Reproducibility stamp:', reproducibility_stamp())

Reproducibility stamp: 92ebdda8929a388f


In [3]:
# 3) Schema Validation Helpers

import numpy as _np

def _broadcast(value, N, dtype=float):
    if isinstance(value, _np.ndarray):
        if value.shape[0] != N:
            raise ValueError(f"Length mismatch: {value.shape[0]} != {N}")
        return value.astype(dtype)
    return _np.full(N, value, dtype=dtype)

def validate_and_canonicalize_samples(X_dict):
    """Broadcast scalars to arrays; clamp within schema bounds; validate enums.
    Returns canonical dict of name->np.ndarray all length N.
    """
    if not isinstance(X_dict, dict):
        raise TypeError("X_dict must be dict")
    # Determine N from first array-like
    N = None
    for v in X_dict.values():
        try:
            N = len(v)
            break
        except Exception:
            continue
    if N is None:
        N = 1
    result = {}
    for name, meta in SCHEMA.items():
        if name in X_dict:
            raw = X_dict[name]
        else:
            raw = meta['default']
        dtype = float if meta['type'] in (float, int) else object
        arr = _broadcast(raw, N, dtype=dtype)
        # Clamp numeric
        if meta['type'] in (float, int):
            if 'min' in meta:
                arr = _np.maximum(arr, meta['min'])
            if 'max' in meta:
                arr = _np.minimum(arr, meta['max'])
        # Validate choices
        if meta['type'] is str and meta.get('choices') is not None:
            choices = set(meta['choices'])
            vals = set(arr.tolist())
            if not vals.issubset(choices):
                raise ValueError(f"Enum violation for {name}: {vals - choices}")
        result[name] = arr
    return result

print("Schema validation helpers loaded.")

Schema validation helpers loaded.


## Gaussian copula: definition, density, properties, usage

### How to run the next cell
- Run the next code cell to construct or refresh the Gaussian copula objects (correlation matrix, Cholesky factor).
- Re-run after changing any dependence or τ-pair settings above.

Definition (Sklar; see Nelsen 2006 https://doi.org/10.1007/978-0-387-28678-0): For CDFs $F_1,\ldots,F_d$ and copula $C:[0,1]^d\to[0,1]$,
$$F_X(x_1,\ldots,x_d) = C\big(F_1(x_1),\ldots,F_d(x_d)\big).$$
Gaussian copula with correlation $\Sigma$:
$$C_{\Sigma}(u) = \Phi_d\big(\Phi^{-1}(u_1),\ldots,\Phi^{-1}(u_d);\,\Sigma\big).$$
Density:
$$c_{\Sigma}(u)= \frac{1}{\sqrt{\det \Sigma}}\exp\!\Big(-\tfrac{1}{2} z^T(\Sigma^{-1}-I)z\Big),\quad z_j=\Phi^{-1}(u_j).$$

Properties: monotone invariance; $\tau=\tfrac{2}{\pi}\arcsin(\rho)$ (Nelsen 2006 https://doi.org/10.1007/978-0-387-28678-0); no tail dependence (contrast with $t$-copulas; Demarta & McNeil 2005 https://doi.org/10.1111/j.1751-5823.2005.tb00254.x). PSD repair via nearest correlation matrices (Higham 2002 https://doi.org/10.1093/imanum/22.3.329).


In [4]:
# 4) Rank-Correlation to PSD Correlation Matrix (tau_to_rho)
# rho = sin(pi * tau / 2); symmetricize; clip eigenvalues; renormalize diagonals

import numpy as _np

def tau_to_rho_matrix(tau_mat):
    tau = _np.asarray(tau_mat, dtype=float)
    if tau.ndim != 2 or tau.shape[0] != tau.shape[1]:
        raise ValueError("tau_to_rho_matrix expects a square matrix")
    rho = _np.sin(_np.pi * tau / 2.0)
    _np.fill_diagonal(rho, 1.0)
    rho = (rho + rho.T) / 2.0
    vals, vecs = _np.linalg.eigh(rho)
    clipped = _np.clip(vals, 1e-12, None)
    repair_norm = float(_np.linalg.norm(vals - clipped))
    if repair_norm > 0:
        rho = (vecs @ _np.diag(clipped) @ vecs.T)
        d = _np.sqrt(_np.clip(_np.diag(rho), 1e-12, None))
        rho = rho / _np.outer(d, d)
        rho = (rho + rho.T) / 2.0
    return rho, repair_norm

print("tau_to_rho_matrix ready (PSD-repaired correlation)")

tau_to_rho_matrix ready (PSD-repaired correlation)


## Flow 4–5: sampling and diagnostics

Transform chain:
$$Z \sim \mathcal N(0,I)\;\Rightarrow\; Z_c=ZL^T\;\Rightarrow\; U=\Phi(Z_c)\;\Rightarrow\; X_j=F_j^{-1}(U_j).$$
Diagnostics: PIT KS statistic $D_n=\sup_u |\hat F_n(u)-u|$ (see Dawid 1984 https://doi.org/10.2307/2982829); KS test (Massey 1951 https://doi.org/10.1080/01621459.1951.10500769); empirical $\tau$ (Kendall 1938 https://doi.org/10.2307/2332226). Degeneracy scan flags zero-variance columns.

### How to run the next cell
- Execute the next code cell to sample joint draws and perform PIT, KS, and Kendall-$\tau$ diagnostics.
- Adjust sample size or correlation settings above, then re-run.
- Inspect printed results for KS p-values and any $\tau$ deviations.

In [5]:
# 5) Physics Kernels (CPU) for DLCZ Metrics
# Computes: losses, T, tau_c, p_link, p_swap, p_succ, R plus legacy comparisons; clips probabilities.

import numpy as _np

def _sanitize_arrays(alpha, L0_km, eta_s, eta_w, eta_r, eta_det, V0, N_modes, n_swaps, kappa_power, c_fiber):
    def clamp(a, lo=None, hi=None):
        if lo is not None: a = _np.maximum(a, lo)
        if hi is not None: a = _np.minimum(a, hi)
        return a
    alpha = clamp(alpha, 0.0)
    L0_km = clamp(L0_km, 1e-9)
    eta_s = clamp(eta_s, 0.0, 1.0)
    eta_w = clamp(eta_w, 0.0, 1.0)
    eta_r = clamp(eta_r, 0.0, 1.0)
    eta_det = clamp(eta_det, 0.0, 1.0)
    V0 = clamp(V0, 0.0, 1.0)
    N_modes = clamp(N_modes, 1.0)
    kappa_power = clamp(kappa_power, 0.0)
    c_fiber = clamp(c_fiber, 1.0)
    n_swaps = _np.maximum(0, _np.rint(n_swaps).astype(int))
    return alpha, L0_km, eta_s, eta_w, eta_r, eta_det, V0, N_modes, n_swaps, kappa_power, c_fiber

def _compute_metrics_cpu(X_dict):
    S = validate_and_canonicalize_samples(X_dict)
    N = len(next(iter(S.values())))
    alpha = S['alpha_dB_per_km'].astype(float)
    L0_km = S['L0_km'].astype(float)
    eta_s = S['eta_s'].astype(float)
    eta_w = S['eta_w'].astype(float)
    eta_r = S['eta_r'].astype(float)
    eta_det = S['eta_det'].astype(float)
    V0 = S['V0'].astype(float)
    N_modes = S['N_modes'].astype(float)
    n_swaps = S['n_swaps'].astype(float)
    kappa_power = S['kappa_power'].astype(float)
    c_fiber = S['c_fiber_mps'].astype(float)
    connector_loss_dB = S.get('connector_loss_dB').astype(float)
    n_connectors = S.get('n_connectors').astype(float)
    splice_loss_dB = S.get('splice_loss_dB').astype(float)
    n_splices = S.get('n_splices').astype(float)
    filter_loss_dB = S.get('filter_loss_dB').astype(float)
    p_dark = S.get('p_dark').astype(float)
    raman_rate_per_ns = S.get('raman_rate_per_ns').astype(float)
    gate_ns = S.get('gate_ns').astype(float)
    geometry = S.get('geometry')
    kappa_mode = S.get('kappa_mode')

    # Extract decoherence time (ms) if provided
    T2_ms = S.get('T2_ms').astype(float) if 'T2_ms' in S else _np.full(N, float(SCHEMA.get('T2_ms',{}).get('default', 150.0)))

    alpha, L0_km, eta_s, eta_w, eta_r, eta_det, V0, N_modes, n_swaps, kappa_power, c_fiber = _sanitize_arrays(
        alpha, L0_km, eta_s, eta_w, eta_r, eta_det, V0, N_modes, n_swaps, kappa_power, c_fiber
    )

    # Loss budget
    L_dB = alpha * L0_km + n_connectors * connector_loss_dB + n_splices * splice_loss_dB + filter_loss_dB
    # Central vs end geometry transmissivity handling
    T_effective = _np.empty(N, dtype=float)
    mask_central = _np.array([g == 'central_bsm' for g in geometry])
    mask_end = ~mask_central
    if mask_end.any():
        T_effective[mask_end] = 10.0 ** (-L_dB[mask_end] / 10.0)
    if mask_central.any():
        L_dB_arm = (alpha[mask_central] * (L0_km[mask_central] / 2.0) + (n_connectors[mask_central] * connector_loss_dB[mask_central] + n_splices[mask_central] * splice_loss_dB[mask_central] + filter_loss_dB[mask_central]) / 2.0)
        T_arm = 10.0 ** (-L_dB_arm / 10.0)
        T_effective[mask_central] = T_arm ** 2

    # Memory efficiency (write * read)
    eta_mem = eta_w * eta_r

    # Elementary link success (corrected) and false-herald suppression
    p_link = 0.5 * eta_s * eta_mem * (eta_det ** 2) * T_effective
    p_link = _np.clip(p_link, 1e-18, 1.0)
    p_false = 1.0 - _np.exp(-(p_dark + raman_rate_per_ns * gate_ns))
    p_link_eff = p_link * (1.0 - p_false)

    # Swap success depending on mode
    p_swap = _np.empty_like(p_link_eff)
    mask_power = _np.array([m == 'power' for m in kappa_mode])
    mask_lin = _np.array([m == 'linear' for m in kappa_mode])
    mask_sq = _np.array([m == 'square' for m in kappa_mode])
    # Implemented mapping: linear -> V0; power/square -> V0**2
    if mask_power.any(): p_swap[mask_power] = V0[mask_power] ** 2.0
    if mask_lin.any(): p_swap[mask_lin] = V0[mask_lin]
    if mask_sq.any(): p_swap[mask_sq] = V0[mask_sq] ** 2.0
    p_swap = _np.clip(p_swap, 1e-12, 1.0)

    # Overall success probability with swap chaining
    p_succ = p_link_eff * (p_swap ** n_swaps)
    p_succ = _np.clip(p_succ, 1e-24, 1.0)

    # Timing & throughput
    # Compute one-way and round-trip times (seconds)
    tau_oneway = (L0_km * 1e3) / _np.maximum(c_fiber, 1e-24)
    tau_c = tau_oneway * 2.0  # full round-trip

    # Decoherence survival: convert T2_ms -> seconds and apply one-way transit survival
    T2_s = _np.maximum(T2_ms, 1e-9) * 1e-3
    survival_factor = _np.exp(-tau_oneway / T2_s)
    # Apply decoherence survival to success probability
    p_succ = p_succ * survival_factor

    R = (N_modes * p_succ) / _np.maximum(tau_c, 1e-24)

    # Legacy expressions (for comparison)
    p_link_legacy = 0.5 * eta_s * eta_mem * (eta_det ** 2) * 10.0 ** (-L_dB / 10.0)
    p_link_legacy = _np.clip(p_link_legacy, 1e-18, 1.0)
    p_swap_legacy = _np.clip(V0 ** kappa_power * (eta_r * eta_det) ** 2 * 0.5, 1e-18, 1.0)
    tau_oneway_legacy = (L0_km * 1e3) / _np.maximum(c_fiber, 1e-24)
    tau_oneway_legacy = tau_oneway_legacy
    tau_oneway_legacy = tau_oneway_legacy
    tau_oneway = tau_oneway
    tau_oneway = tau_oneway
    tau_oneway = tau_oneway
    tau_oneway = tau_oneway
    tau_oneway = tau_oneway
    tau_oneway = tau_oneway
    tau_oneway = tau_oneway
    tau_oneway = tau_oneway
    tau_oneway = tau_oneway
    tau_oneway = tau_oneway
    # Keep legacy timing computation for T_avg
    tau_oneway = (L0_km * 1e3) / _np.maximum(c_fiber, 1e-24)
    T_avg_legacy = tau_oneway / _np.maximum(N_modes * p_link_legacy, 1e-24) * (1.0 / _np.maximum(p_swap_legacy, 1e-24)) ** n_swaps
    R_legacy = 1.0 / _np.maximum(T_avg_legacy, 1e-30)

    return {
        'T_effective': T_effective,
        'tau_c': tau_c,
        'p_link': p_link_eff,
        'p_swap': p_swap,
        'p_succ': p_succ,
        'R': R,
        'p_link_legacy': p_link_legacy,
        'p_swap_legacy': p_swap_legacy,
        'R_legacy': R_legacy,
        'T_avg_legacy': T_avg_legacy,
    }

print("CPU physics kernels loaded.")


CPU physics kernels loaded.


## From Kendall $\tau$ to Gaussian $\rho$; PSD fix and Cholesky

Mapping ($\tau\to\rho$ Gaussian relation: Nelsen 2006 https://doi.org/10.1007/978-0-387-28678-0):
$$
\rho_{ij} = \sin\Big(\frac{\pi}{2}\,\tau_{ij}\Big),\quad \rho_{ii}=1.
$$

PSD repair (nearest correlation; Higham 2002 https://doi.org/10.1093/imanum/22.3.329):
$$
R = Q\Lambda Q^T\;\Rightarrow\;\Lambda' = \max(\Lambda,\,\epsilon I)\;\Rightarrow\;\Sigma=Q\Lambda'Q^T,\;\operatorname{diag}(\Sigma)\leftarrow 1,\;\Sigma\leftarrow\tfrac{\Sigma+\Sigma^T}{2}.
$$

Cholesky with jitter: find minimal $\delta\ge 0$ so that $\operatorname{chol}(\Sigma+\delta I)$ exists; start from small $\delta$ and grow geometrically.

### How to run the next cell
- Run the following code cell to construct/repair the correlation matrix and Cholesky factor.
- If you edited τ-pairs or dependence, do that first, then press (▶) or Shift+Enter.
- Eigenvalues produced; re-run after any edits.

In [6]:
# Gaussian copula builder per user's contract
import numpy as np
from typing import List, Tuple, Dict, Any, Optional


def build_copula(VAR_ORDER: List[str],
                 kendall_tau_pairs: Optional[List[Tuple[str, str, float]]] = None,
                 Sigma_source: Optional[np.ndarray] = None,
                 options: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Build a Gaussian copula for sampling.

    Returns a CopulaResult dict with keys: Sigma_psd, L, diag.

    Contract:
      - VAR_ORDER: frozen list of variable keys (sampling order)
      - kendall_tau_pairs: list of (key_i, key_j, tau_ij)
      - Sigma_source: optional full correlation matrix (ordered to VAR_ORDER)
      - options: epsilon_psd, jitter_chol, fallback_on_empty_tau
    """
    opts = {
        "epsilon_psd": 1e-12,
        "jitter_chol": 1e-12,
        "fallback_on_empty_tau": "identity",
        "diag_tol": 1e-12,
    }
    if options:
        opts.update(options)

    d = len(VAR_ORDER)
    name_to_idx = {k: i for i, k in enumerate(VAR_ORDER)}

    # Validation helpers
    def _validate_key(k):
        if k not in name_to_idx:
            raise ValueError(f"Copula key '{k}' not in VAR_ORDER")

    # Build raw rho matrix either from Sigma_source or from kendall tau pairs
    fallback = "none"
    if Sigma_source is not None:
        # Validate Sigma_source shape
        A = np.asarray(Sigma_source, dtype=float)
        if A.shape != (d, d):
            raise ValueError(f"Sigma_source must be {d}x{d}")
        # Symmetry check
        if not np.allclose(A, A.T, atol=1e-12):
            raise ValueError("Sigma_source is not symmetric")
        # Diagonal ones
        diag = np.diag(A)
        if not np.allclose(diag, np.ones(d), atol=opts["diag_tol"]):
            raise ValueError("Sigma_source diagonal is not (approximately) 1")
        rho_raw = (A + A.T) / 2.0
        n_pairs = int(d * (d - 1) / 2)
        avg_abs_tau = 0.0
    else:
        tau_mat = np.zeros((d, d), dtype=float)
        n_pairs = 0
        taus = []
        if kendall_tau_pairs:
            seen = set()
            for (a, b, tau) in kendall_tau_pairs:
                _validate_key(a)
                _validate_key(b)
                if a == b:
                    raise ValueError("Self-pair provided in kendall_tau_pairs")
                if not np.isfinite(tau) or abs(tau) >= 1.0:
                    raise ValueError(f"Invalid tau value for pair {(a,b)}: {tau}")
                i = name_to_idx[a]
                j = name_to_idx[b]
                if (i, j) in seen or (j, i) in seen:
                    raise ValueError(f"Duplicate pair {(a,b)}")
                seen.add((i, j))
                tau_mat[i, j] = tau
                tau_mat[j, i] = tau
                taus.append(tau)
                n_pairs += 1
        # handle empty pairs
        if n_pairs == 0:
            if opts["fallback_on_empty_tau"] == "identity":
                fallback = "identity"
                rho_raw = np.eye(d)
                avg_abs_tau = 0.0
            else:
                raise ValueError("No kendall_tau_pairs provided and fallback_on_empty_tau != 'identity'")
        else:
            avg_abs_tau = float(np.mean(np.abs(taus)))
            # Map tau -> rho
            rho_raw = np.sin(0.5 * np.pi * tau_mat)
            np.fill_diagonal(rho_raw, 1.0)

    # Force symmetry
    rho_raw = 0.5 * (rho_raw + rho_raw.T)
    np.fill_diagonal(rho_raw, 1.0)

    # PSD repair via eigenvalue clipping
    eigvals, eigvecs = np.linalg.eigh(rho_raw)
    clipped = np.maximum(eigvals, opts["epsilon_psd"])
    min_eig_after_clip = float(np.min(clipped))
    max_eig_after_clip = float(np.max(clipped))

    Sigma_psd = (eigvecs @ np.diag(clipped) @ eigvecs.T)
    # Reunit-diagonal to correct tiny drift and re-symmetrize
    diag_before = np.diag(Sigma_psd).copy()
    np.fill_diagonal(Sigma_psd, 1.0)
    Sigma_psd = 0.5 * (Sigma_psd + Sigma_psd.T)

    psd_correction_norm = float(np.linalg.norm(rho_raw - Sigma_psd, ord='fro'))

    # Attempt Cholesky with jitter
    jitter = float(opts.get("jitter_chol", 1e-12))
    L = None
    chol_jitter_used = 0.0
    try:
        L = np.linalg.cholesky(Sigma_psd + jitter * np.eye(d))
        chol_jitter_used = jitter
    except np.linalg.LinAlgError:
        # try increasing jitter a few times
        for factor in [1e-8, 1e-6, 1e-4, 1e-2]:
            try_j = jitter * factor
            try:
                L = np.linalg.cholesky(Sigma_psd + try_j * np.eye(d))
                chol_jitter_used = try_j
                break
            except np.linalg.LinAlgError:
                continue
        if L is None:
            raise RuntimeError("Cholesky failed even after adding jitter; Sigma_psd not positive-definite")

    # Logging as required by contract
    print(f"[copula] n_pairs={n_pairs}, avg_abs_tau={avg_abs_tau:.4g}")
    print(f"[copula] min_eig_after_clip={min_eig_after_clip:.4g}, max_eig_after_clip={max_eig_after_clip:.4g}")
    print(f"[copula] psd_correction_norm={psd_correction_norm:.4g}")

    # Print labeled 3x3 block (first 3 vars)
    n_show = min(3, d)
    labels = VAR_ORDER[:n_show]
    block = Sigma_psd[:n_show, :n_show]
    # Nicely print header
    header = "\t" + "\t".join(labels)
    print("[copula] first %d×%d block of Sigma_psd:" % (n_show, n_show))
    print(header)
    for i, lab in enumerate(labels):
        row = "\t".join([f"{v: .4g}" for v in block[i, :n_show]])
        print(f"{lab}\t{row}")

    if fallback != "none":
        print(f"[copula] fallback used: {fallback}")

    diag = {
        "n_pairs": int(n_pairs),
        "avg_abs_tau": float(avg_abs_tau),
        "psd_correction_norm": float(psd_correction_norm),
        "min_eig_after_clip": float(min_eig_after_clip),
        "max_eig_after_clip": float(max_eig_after_clip),
        "fallback": fallback,
        "chol_jitter_used": float(chol_jitter_used),
    }

    return {"Sigma_psd": Sigma_psd, "L": L, "diag": diag}


### DLCZ in one minute (for this notebook)

- Each elementary link tries to create a single spin-wave + Stokes-photon pair with small probability (write step), then heralds entanglement via a central interference and detection event.
- Successful heralds populate $p_{\text{link}}$; spurious clicks from dark/Raman populate $p_{\text{false}}$.
- Entanglement swapping stitches links at repeater nodes with success $p_{\text{swap}}$, compounding across segments.
- Memories keep spin waves alive over classical signaling and reattempt cycles with survival $s=\exp(-\tau_{\text{oneway}}/T_2)$.
- Overall rate combines success probability, survival, and multiplexed modes: $$R=\dfrac{N_{\text{modes}}\, p_{\text{succ}}\, s}{\tau_c}.$$

### How to use the next cell

- Execute the next code cell (Shift+Enter or ▶) to run this section’s computations.
- Adjust relevant settings above first; re-run after any changes.
- Review the output and any warnings below.

In [7]:
# 6) Physics Kernels (Torch/Device) and Device-Aware Wrapper
# Uses float32 tensors; falls back to CPU if device unsupported.

HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
except Exception:
    torch = None
    HAS_TORCH = False


def compute_metrics_torch(X_dict, device):
    if (not HAS_TORCH) or str(device).lower() in (None, 'cpu'):
        return _compute_metrics_cpu(X_dict)
    dev = torch.device(device)
    S = validate_and_canonicalize_samples(X_dict)
    N = len(next(iter(S.values())))
    # Helper to move/broadcast
    def tfill(x):
        return torch.full((N,), float(x), device=dev, dtype=torch.float32)

    # Scalars from medians of arrays (device path is illustrative; CPU path preferred)
    def med(name):
        a = S[name]
        return float(_np.median(a))

    alpha = med('alpha_dB_per_km')
    L0_km = med('L0_km')
    eta_s = med('eta_s')
    eta_w = med('eta_w')
    eta_r = med('eta_r')
    eta_det = med('eta_det')
    V0 = med('V0')
    N_modes = med('N_modes')
    n_swaps = int(round(med('n_swaps')))
    kappa_power = med('kappa_power')
    c_fiber = med('c_fiber_mps')
    connector_loss_dB = med('connector_loss_dB')
    n_connectors = int(round(med('n_connectors')))
    splice_loss_dB = med('splice_loss_dB')
    n_splices = int(round(med('n_splices')))
    filter_loss_dB = med('filter_loss_dB')

    alpha_t = tfill(alpha)
    L0_t = tfill(L0_km)
    eta_s_t = tfill(eta_s)
    eta_w_t = tfill(eta_w)
    eta_r_t = tfill(eta_r)
    eta_det_t = tfill(eta_det)
    V0_t = tfill(V0)
    N_modes_t = tfill(N_modes)
    n_swaps_t = tfill(n_swaps)
    kappa_power_t = tfill(kappa_power)
    c_fiber_t = tfill(c_fiber)

    L_dB = alpha_t * L0_t + n_connectors * connector_loss_dB + n_splices * splice_loss_dB + filter_loss_dB
    T_full = torch.pow(10.0, -L_dB / 10.0)

    eta_mem_t = eta_w_t * eta_r_t

    p_link_new = 0.5 * eta_s_t * eta_mem_t * (eta_det_t ** 2) * T_full
    p_link_new = torch.clamp(p_link_new, 1e-18, 1.0)

    p_swap_new = torch.pow(V0_t, kappa_power_t)
    p_swap_new = torch.clamp(p_swap_new, 1e-12, 1.0)

    p_succ = p_link_new * torch.pow(p_swap_new, n_swaps_t)
    p_succ = torch.clamp(p_succ, 1e-24, 1.0)

    tau_c = (L0_t * 1e3 * 2.0) / torch.clamp(c_fiber_t, 1e-24)
    R = (N_modes_t * p_succ) / torch.clamp(tau_c, 1e-24)

    return {
        'R': R.cpu().numpy(),
    }


def run_on_device(cfg_df, sampleN, device_str='cpu'):
    """Sample marginals from a config DataFrame and run kernels on selected device.
    Returns (sampled_dict, metrics_dict, elapsed_seconds).
    """
    import time as _time
    sampled = {}
    rows = cfg_df.to_dict(orient='records')
    for row in rows:
        key = row.get('variable') or row.get('name') or row.get('key')
        sampled[key] = sample_marginal_inline(row, sampleN, rng=RNG)
    start = _time.time()
    if (not HAS_TORCH) or device_str in (None, 'cpu', 'numpy'):
        metrics = _compute_metrics_cpu(sampled)
    else:
        try:
            metrics = compute_metrics_torch(sampled, device_str)
        except Exception as e:
            print('Device path failed -> CPU fallback. Error:', e)
            metrics = _compute_metrics_cpu(sampled)
    elapsed = _time.time() - start
    return sampled, metrics, elapsed

print(f"Torch available: {HAS_TORCH}")

Torch available: False


In [8]:
# 7) Inline Marginal Sampler (const/normal/lognormal/uniform/beta)

import numpy as _np

def sample_marginal_inline(row, N, rng=None):
    if rng is None:
        rng = _np.random.default_rng()
    if not isinstance(row, dict):
        raise TypeError("row must be dict with keys: variable/dist/params")
    dist = (row.get('dist') or '').lower()
    params = row.get('params') or {}
    if dist == 'const':
        val = float(params.get('value', 0.0))
        return _np.full(N, val, dtype=float)
    if dist == 'normal':
        mu = float(params.get('mu', 0.0)); sigma = max(1e-12, float(params.get('sigma', 1.0)))
        return rng.normal(mu, sigma, size=N)
    if dist == 'lognormal':
        mu = float(params.get('mu', 0.0)); sigma = max(1e-12, float(params.get('sigma', 1.0)))
        return rng.lognormal(mean=mu, sigma=sigma, size=N)
    if dist == 'uniform':
        low = float(params.get('low', 0.0)); high = float(params.get('high', 1.0))
        if high < low: low, high = high, low
        return rng.uniform(low, high, size=N)
    if dist == 'beta':
        a = float(params.get('a', 1.0)); b = float(params.get('b', 1.0))
        low = float(params.get('low', 0.0)); high = float(params.get('high', 1.0))
        raw = rng.beta(a, b, size=N)
        return low + (high - low) * raw
    raise ValueError(f"Unsupported dist family: {dist}")

print("Marginal sampler ready.")

Marginal sampler ready.


# Supported marginal distributions: PDFs, CDFs, and shapes

We support several families; choose the one matching semantics (boundedness, skew). Below are the key formulas and typical shapes.

- Beta(a,b) on [L,H] (bounded proportion modeling; regularized incomplete beta; Wilson interval contexts Wilson 1927 https://doi.org/10.2307/2280061)
  - Transform: draw Y ~ Beta(a,b) on [0,1], then X = L + (H−L)Y.
  - PDF on [0,1]: \(f_Y(y) = \frac{y^{a-1}(1-y)^{b-1}}{B(a,b)}\); CDF uses regularized incomplete beta \(I_y(a,b)\).
  - Shapes: for a,b>1 bell-shaped; a<1 or b<1 creates J/U-shapes.

- Lognormal(μ,σ) (heavy right tail; multiplicative processes; Limpert et al. 2001 https://doi.org/10.1002/1521-186X(200112)22:6<636::AID-BIMB636>3.0.CO;2-L)
  - If Z ~ N(μ,σ²) then X = e^Z.
  - PDF: \(f(x) = \frac{1}{x\,\sigma\sqrt{2\pi}}\exp\!\big(-\tfrac{(\ln x - \mu)^2}{2\sigma^2}\big),\ x>0\).
  - Right-skewed; median = e^μ; heavier tail with larger σ.

- Normal(μ,σ) (central limit archetype; Gauss 1809 historical)
  - PDF: \(f(x) = \frac{1}{\sigma\sqrt{2\pi}}\exp\!\big(-\tfrac{(x-\mu)^2}{2\sigma^2}\big)\).
  - Symmetric, unbounded; use only when negative values are acceptable/meaningful.

- Uniform([L,H]) (maximum entropy on finite interval)
  - PDF: f(x)=1/(H−L) on [L,H]; 0 elsewhere.

- Truncated Normal([L,H]; μ,σ) (tail-pruned Gaussian; truncated distribution theory)
  - Derived from N(μ,σ²) restricted to [L,H]; use SciPy `truncnorm` with normalized a=(L−μ)/σ, b=(H−μ)/σ. PPF maps u∈[0,1] to [L,H].
  - Appropriate when values concentrate around μ but cannot leave [L,H].

- Constant(value)
  - Degenerate distribution at a point; zero variance; excluded from PIT KS tests; useful for fixed known parameters.

- Empirical (ECDF-based) (nonparametric interpolation; Silverman 1986 KDE context https://doi.org/10.1007/978-1-4899-3324-9)
  - Built from samples using ECDF and inverse CDF interpolation; captures arbitrary shapes from data.

Implementation detail: CDF/PPF come from SciPy (`beta`, `lognorm`, `norm`, `uniform`, `truncnorm`) and are used elementwise in the copula sampling pipeline.


### Marginal Configuration UI — detailed guide

The next cell opens an interactive panel to define the probability distributions ("marginals") for each model variable and to manage presets.

#### Left pane controls
- Variable: pick a variable to edit (units and bounds come from the internal schema).
- dist: choose a family:
  - const: deterministic value `value`.
  - normal: parameters `mu`, `sigma` (unbounded).
  - lognormal: parameters are log-domain `mu`, `sigma` (so X = exp(mu + sigma·Z)).
  - uniform: `low`, `high` (inclusive bounds).
  - beta: `a`, `b`, scaled to [`low`,`high`].
- Param fields: numeric boxes for the chosen family; editing them updates the working config and the preview.
- Overlay defaults: when checked, overlays a histogram of the default distribution for comparison.
- Reset defaults: restores all variables to the built-in defaults (see schema).

#### Presets (save/load/delete)
- Preset: enter a name then click 'Save preset' to write `config/presets/<name>.json`.
  - Preset files store a dict keyed by variable with `{dist, params}` for quick reuse.
- Load: pick a preset from the dropdown then click 'Load' to replace the current working marginals.
- Delete: removes the selected preset file from `config/presets/`.
- Save config: writes a standardized `config/marginal_config.json` used by runners/flows. This contains a list of entries with `key`, `unit`, `family`, `params`, and optional `bounds`.
- Status: messages appear below the buttons (green = success, red = error).

#### Right pane (preview)
- Live histogram of the currently selected variable with the chosen distribution and parameters.
- With 'Overlay defaults' on, a second histogram shows the built-in default for quick visual comparison.

#### Typical workflow
1) Select a variable → choose `dist` → set parameters; confirm the shape in the preview.
2) Repeat for other variables via the Variable list.
3) Click 'Save config' to export `config/marginal_config.json` for subsequent runs.
4) Optionally click 'Save preset' (with a name) to keep a versioned snapshot under `config/presets/`.
5) Later, use 'Load' to quickly restore a saved preset.

Notes:
- For lognormal, remember `mu`,`sigma` are in log space; the median is exp(mu).
- For beta scaled, ensure `low < high` and `a,b > 0`; the shape can be U-like (a,b < 1) or bell-like (a,b > 1).
- Bounds shown in previews are informational; if schema bounds exist, they're recorded in the saved config.

In [9]:
# 10) Marginal Configuration UI (presets, save/load, Plotly preview)
# Idempotent; closes previous widgets; overlays defaults; persists to config/marginal_config.json
# Extended: default values aligned to SCHEMA where numeric; includes additional noise terms;
# Provides get_marginal_config() for external runners. Stable parameter ordering retained.

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import json as _json
    import numpy as _np
    try:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        _HAS_PLOTLY = True
    except Exception:
        _HAS_PLOTLY = False

    # Close previous UI if any
    _MUI = globals().get('_MARGINAL_UI_WIDGETS')
    if _MUI:
        for w in _MUI:
            try: w.close()
            except Exception: pass
    _MUI = []

    # Defaults aligned with SCHEMA numeric baselines (plus explicit noise terms)
    DEFAULT_MARGINALS = {
        'alpha_dB_per_km': {'dist': 'normal', 'params': {'mu': 0.2, 'sigma': 0.02}},
        'L0_km': {'dist': 'normal', 'params': {'mu': 50.0, 'sigma': 5.0}},
        'eta_s': {'dist': 'beta', 'params': {'a': 9.0, 'b': 1.0, 'low': 0.5, 'high': 1.0}},
        'eta_w': {'dist': 'beta', 'params': {'a': 9.0, 'b': 1.0, 'low': 0.5, 'high': 1.0}},
        'eta_r': {'dist': 'beta', 'params': {'a': 9.0, 'b': 1.0, 'low': 0.5, 'high': 1.0}},
        'eta_det': {'dist': 'beta', 'params': {'a': 17.0, 'b': 3.0, 'low': 0.7, 'high': 0.99}},
        'V0': {'dist': 'beta', 'params': {'a': 8.0, 'b': 2.0, 'low': 0.7, 'high': 1.0}},
        'N_modes': {'dist': 'const', 'params': {'value': 50}},
        'n_swaps': {'dist': 'const', 'params': {'value': 3}},
        'kappa_power': {'dist': 'const', 'params': {'value': 1.0}},
        'c_fiber_mps': {'dist': 'const', 'params': {'value': 2.0e8}},
        'n_connectors': {'dist': 'const', 'params': {'value': 2}},
        'connector_loss_dB': {'dist': 'const', 'params': {'value': 0.5}},
        'n_splices': {'dist': 'const', 'params': {'value': 5}},
        'splice_loss_dB': {'dist': 'const', 'params': {'value': 0.1}},
        'filter_loss_dB': {'dist': 'const', 'params': {'value': 0.0}},
        'p_dark': {'dist': 'const', 'params': {'value': 0.0}},
        'raman_rate_per_ns': {'dist': 'const', 'params': {'value': 0.0}},
        'gate_ns': {'dist': 'const', 'params': {'value': 1.0}},
    }
    # Ensure WORKING_MARGINALS exists and follows the default shape
    WORKING_MARGINALS = globals().get('WORKING_MARGINALS') or {k:{'dist':v.get('dist') or v.get('family') or 'const','params':dict(v.get('params',{}))} for k,v in DEFAULT_MARGINALS.items()}

    DIST_TYPES = {
        'const': ['value'],
        'normal': ['mu','sigma'],
        'lognormal': ['mu','sigma'],
        'uniform': ['low','high'],
        'beta': ['a','b','low','high'],
    }

    def _label(var):
        return f"{var}"

    def _sample_family(dist, params, N=16000, rng=None):
        return sample_marginal_inline({'dist':dist,'params':params}, N, rng=rng or _np.random.default_rng(123))

    def _get_dist(spec):
        # Accept either 'dist' or 'family' keys
        if not isinstance(spec, dict):
            return 'normal'
        return spec.get('dist') or spec.get('family') or 'normal'

    shared_plot_out = widgets.Output(layout=widgets.Layout(width='640px', border='1px solid #ddd'))
    post_out = widgets.Output()

    var_list = list(WORKING_MARGINALS.keys())
    var_select = widgets.Select(options=[(_label(k),k) for k in var_list], value=var_list[0], description='Variable', rows=10)
    current_dist_dd = widgets.Dropdown(options=list(DIST_TYPES.keys()), value=_get_dist(WORKING_MARGINALS[var_list[0]]), description='dist:')
    params_box = widgets.VBox([])
    overlay_chk = widgets.Checkbox(value=True, description='Overlay defaults')

    preset_name_txt = widgets.Text(description='Preset:', placeholder='e.g. high_loss', layout=widgets.Layout(width='260px'))
    save_preset_btn = widgets.Button(description='Save preset')
    load_preset_dd = widgets.Dropdown(options=['(none)'], description='Load:')
    load_preset_btn = widgets.Button(description='Load')
    delete_preset_btn = widgets.Button(description='Delete', button_style='danger')
    save_btn = widgets.Button(description='Save config', button_style='success')
    reset_btn = widgets.Button(description='Reset defaults', button_style='warning')
    status_msg = widgets.HTML(value='')

    PRESETS_DIR = CONFIG_DIR / 'presets'

    current_params_widgets = {}

    def _list_presets():
        return [p.stem for p in PRESETS_DIR.glob('*.json')]

    def _refresh_presets():
        load_preset_dd.options = ['(none)'] + _list_presets()

    def _update_working(var):
        # Store both 'dist' and 'family' for compatibility with other code variants
        WORKING_MARGINALS[var] = {
            'dist': current_dist_dd.value,
            'family': current_dist_dd.value,
            'params': {k: float(w.value) for k, w in current_params_widgets.items()}
        }

    def _rebuild_param_widgets(var):
        global current_params_widgets
        current_params_widgets = {}
        spec = WORKING_MARGINALS.get(var, {'dist':'normal','params':{'mu':0,'sigma':1}})
        dist_name = _get_dist(spec)
        current_dist_dd.value = dist_name if dist_name in DIST_TYPES else 'normal'
        items = []
        # Only build widgets for current dist type (stable order)
        for p in DIST_TYPES[current_dist_dd.value]:
            init_raw = spec.get('params',{}).get(p, 1.0 if p=='sigma' else 0.0)
            try:
                init = float(init_raw)
            except Exception:
                init = 0.0
            w = widgets.FloatText(value=init, description=p+':', layout=widgets.Layout(width='180px'))
            def _mk(param_key):
                def _chg(ch):
                    if ch['name']=='value':
                        try: float(ch['new'])
                        except Exception: return
                        _update_working(var_select.value)
                        _render()
                return _chg
            w.observe(_mk(p), names='value')
            current_params_widgets[p] = w
            items.append(w)
        params_box.children = items

    def _current_params():
        out = {}
        for k,w in current_params_widgets.items():
            try: out[k] = float(w.value)
            except Exception: out[k] = 0.0
        return out

    def _render():
        with shared_plot_out:
            clear_output(wait=True)
            var = var_select.value
            params = _current_params()
            data = _sample_family(current_dist_dd.value, params, N=16000, rng=_np.random.default_rng(42))
            if not _HAS_PLOTLY:
                print('Plotly unavailable.')
                return
            fig = go.Figure()
            nbins = 90 if data.size>8000 else max(30, int(data.size//80))
            fig.add_trace(go.Histogram(x=data, nbinsx=nbins, name='Current', opacity=0.7))
            if overlay_chk.value:
                dft = DEFAULT_MARGINALS.get(var)
                if dft:
                    data2 = _sample_family(_get_dist(dft), dft.get('params',{}), N=16000, rng=_np.random.default_rng(7))
                    fig.add_trace(go.Histogram(x=data2, nbinsx=nbins, name='Default', opacity=0.4))
            fig.update_layout(title=f"{var} — {current_dist_dd.value}({_current_params()})", barmode='overlay', bargap=0.03, width=650, height=380)
            display(fig)

    def _on_var(ch):
        if ch['name']=='value' and ch['new']:
            _rebuild_param_widgets(ch['new'])
            _render()

    def _on_dist(ch):
        if ch['name']=='value':
            # Rebuild widgets if distribution family changed
            _rebuild_param_widgets(var_select.value)
            _update_working(var_select.value)
            _render()

    def _on_reset(_):
        for k,v in DEFAULT_MARGINALS.items():
            WORKING_MARGINALS[k] = {'dist': v.get('dist') or v.get('family'), 'params': dict(v.get('params',{}))}
        _rebuild_param_widgets(var_select.value)
        _render()
        status_msg.value = '<span style="color:#d9534f">Reset to defaults.</span>'

    def _on_save_cfg(_):
        marg_list = []
        for k,v in WORKING_MARGINALS.items():
            dist_name = _get_dist(v)
            unit = SCHEMA.get(k,{}).get('unit','')
            params = v.get('params',{})
            bounds = None
            lo = SCHEMA.get(k,{}).get('min', None)
            hi = SCHEMA.get(k,{}).get('max', None)
            if lo is not None or hi is not None:
                bounds = [lo, hi]
            entry = {'key': k, 'unit': unit, 'family': (dist_name.title() if dist_name else dist_name), 'params': dict(params)}
            if bounds is not None:
                entry['bounds'] = bounds
            marg_list.append(entry)
        try:
            PRESETS_DIR.mkdir(parents=True, exist_ok=True)
            with open(CONFIG_DIR / 'marginal_config.json','w',encoding='utf-8') as f:
                _json.dump({'marginals': marg_list}, f, indent=2)
            status_msg.value = f"<span style='color:#5cb85c'>Saved: {CONFIG_DIR / 'marginal_config.json'}</span>"
        except Exception as e:
            status_msg.value = f"<span style='color:#d9534f'>Save failed: {e}</span>"
        with post_out:
            clear_output(wait=True)
            if _HAS_PLOTLY:
                items = list(WORKING_MARGINALS.items())
                cols=3; rows=(len(items)+cols-1)//cols
                fig2 = make_subplots(rows=rows, cols=cols, subplot_titles=[k for k,_ in items])
                for i,(nm, spec) in enumerate(items):
                    r=i//cols+1; c=i%cols+1
                    arr = _sample_family(_get_dist(spec), spec.get('params',{}), N=8000, rng=_np.random.default_rng(77+i))
                    nb=max(30,int(arr.size//120))
                    fig2.add_trace(go.Histogram(x=arr, nbinsx=nb, showlegend=False), r,c)
                fig2.update_layout(title_text='Configured marginals (preview)', height=rows*240, bargap=0.03)
                display(fig2)
            else:
                print('Plotly unavailable for save preview.')

    def _on_save_preset(_):
        name = (preset_name_txt.value or '').strip()
        if not name:
            status_msg.value = '<span style="color:#d9534f">Preset name required.</span>'
            return
        PRESETS_DIR.mkdir(parents=True, exist_ok=True)
        try:
            with open(PRESETS_DIR / f'{name}.json','w',encoding='utf-8') as f:
                _json.dump({'marginals': WORKING_MARGINALS}, f, indent=2)
            status_msg.value = f"<span style='color:#5cb85c'>Preset saved: {name}</span>"
        except Exception as e:
            status_msg.value = f"<span style='color:#d9534f'>Preset save failed: {e}</span>"
        _refresh_presets()

    def _on_load_preset(_):
        sel = load_preset_dd.value
        if sel in (None,'(none)'):
            status_msg.value = '<span style="color:#d9534f">No preset selected.</span>'
            return
        path = PRESETS_DIR / f'{sel}.json'
        try:
            obj = _json.load(open(path,'r',encoding='utf-8'))
            m = obj.get('marginals', {})
            if isinstance(m, dict):
                for k,v in m.items():
                    WORKING_MARGINALS[k] = {'dist': v.get('dist') or v.get('family','normal'), 'params': dict(v.get('params',{}))}
            status_msg.value = f"<span style='color:#5cb85c'>Loaded preset: {sel}</span>"
            _rebuild_param_widgets(var_select.value)
            _render()
        except Exception as e:
            status_msg.value = f"<span style='color:#d9534f'>Load failed: {e}</span>"

    def _on_delete_preset(_):
        sel = load_preset_dd.value
        if sel in (None,'(none)'):
            status_msg.value = '<span style="color:#d9534f">No preset selected.</span>'
            return
        p = PRESETS_DIR / f'{sel}.json'
        try:
            if p.exists():
                p.unlink()
                status_msg.value = f"<span style='color:#5cb85c'>Deleted preset: {sel}</span>"
                _refresh_presets()
            else:
                status_msg.value = '<span style="color:#d9534f">Preset file missing.</span>'
        except Exception as e:
            status_msg.value = f"<span style='color:#d9534f'>Delete failed: {e}</span>"

    var_select.observe(_on_var, names='value')
    current_dist_dd.observe(_on_dist, names='value')
    overlay_chk.observe(lambda ch: _render() if ch['name']=='value' else None, names='value')
    reset_btn.on_click(_on_reset)
    save_btn.on_click(_on_save_cfg)
    save_preset_btn.on_click(_on_save_preset)
    load_preset_btn.on_click(_on_load_preset)
    delete_preset_btn.on_click(_on_delete_preset)

    # Initial render
    _refresh_presets()
    _rebuild_param_widgets(var_select.value)
    _render()

    left = widgets.VBox([
        widgets.HTML('<b>Marginal configuration</b>'),
        var_select, current_dist_dd, params_box,
        widgets.HBox([overlay_chk, reset_btn]),
        widgets.HTML('<b>Presets</b>'),
        widgets.HBox([preset_name_txt, save_preset_btn]),
        widgets.HBox([load_preset_dd, load_preset_btn, delete_preset_btn]),
        widgets.HBox([save_btn]),
        status_msg,
    ])
    ui = widgets.HBox([left, shared_plot_out])
    display(ui)
    display(post_out)

    def get_marginal_config():
        """Return current working marginals as list-of-dicts (variable/key/family/params) per final spec."""
        marg_list = []
        for k,v in WORKING_MARGINALS.items():
            dist_name = _get_dist(v)
            unit = SCHEMA.get(k,{}).get('unit','')
            params = dict(v.get('params',{}))
            entry = {'key': k, 'unit': unit, 'family': (dist_name.title() if dist_name else dist_name), 'params': params}
            lo = SCHEMA.get(k,{}).get('min', None)
            hi = SCHEMA.get(k,{}).get('max', None)
            if lo is not None or hi is not None:
                entry['bounds'] = [lo, hi]
            marg_list.append(entry)
        return marg_list

    _MUI.extend([ui, post_out, left, var_select, current_dist_dd, params_box, overlay_chk, reset_btn, preset_name_txt, save_preset_btn, load_preset_dd, load_preset_btn, delete_preset_btn, save_btn, status_msg, shared_plot_out])
    globals()['_MARGINAL_UI_WIDGETS'] = _MUI
except Exception as e:
    print('Marginal config UI unavailable:', e)


Output()

## Physics kernels (derivation sketch)

### How to run the next cell (Cell 20)
- Execute the next code cell to compute physics-derived metrics using current sampled parameters.
- Re-run after new sampling or parameter changes to refresh metrics.

DLCZ link and repeater chain are modeled with coarse-grained probabilities (Duan et al. 2001 Nature https://doi.org/10.1038/35106500; Briegel et al. 1998 PRL https://doi.org/10.1103/PhysRevLett.81.5932) capturing write/read efficiencies, losses, detector QE, and noise.

- Loss budget (dB): $$L_{\text{dB}} = \alpha L_0 + n_c L_c + n_s L_s + L_f.$$
- Transmissivity: $$T = 10^{-L_{\text{dB}}/10}.$$
- Heralded link success (central BSM): $$p_{\text{link}} \approx \tfrac{1}{2}\,\eta_s\eta_w\eta_r\eta_{\text{det}}^2\,T_{\text{eff}}.$$
- False window probability (Raman/EIT context: Fleischhauer et al. 2005 RMP https://doi.org/10.1103/RevModPhys.77.633): $$p_{\text{false}} = 1-\exp\big(- (p_{\text{dark}} + r_{\text{raman}}\,t_{\text{gate}})\big).$$
- Effective link success: $$p_{\text{link,eff}} \approx p_{\text{link}} (1-p_{\text{false}}).$$
- Swapping: $$p_{\text{succ}} = p_{\text{link,eff}}\, p_{\text{swap}}^{n_{\text{swaps}}}.$$
- Memory survival (Ramsey decoherence: Ramsey 1950 Phys. Rev. https://doi.org/10.1103/PhysRev.78.695): $$s=\exp(-\tau_{\text{oneway}}/T_2).$$
- Throughput: $$R = \dfrac{N_{\text{modes}}\, p_{\text{succ}}\, s}{\tau_c}.$$


In [10]:
# Flow Step 0–3 (Refactored): Bootstrap → Config validation → Marginal binding → Copula assembly
# Reuses SCHEMA defaults when CONFIG missing; treats CONFIG as overlay.

try:
    import numpy as _np, json as _json, hashlib as _hashlib
    from pathlib import Path as _Path
    from math import pi
    from scipy.stats import beta as _beta, lognorm as _lognorm, uniform as _uniform, truncnorm as _truncnorm
    from scipy.stats import norm as _norm

    if 'FLOW_STATE' not in globals():
        FLOW_STATE = {}

    # Bootstrap
    device = str(globals().get('SELECTED_DEVICE', 'cpu') or 'cpu')
    dtype = str(_np.array(0.0).dtype)
    seed = int(globals().get('GLOBAL_SEED', 123456789))
    print(f"[Flow 0] seed={seed} device={device} dtype={dtype}")
    FLOW_STATE.update({'seed': seed, 'device': device, 'dtype': dtype, 'warnings': []})

    # Merge CONFIG over SCHEMA defaults
    raw_cfg = globals().get('CONFIG') or {}
    cfg = {}
    for k, meta in SCHEMA.items():
        cfg[k] = raw_cfg.get(k, meta.get('default'))
    # Nested mc
    mc_cfg = raw_cfg.get('mc', {})
    if 'mc_samples' in raw_cfg and 'samples' not in mc_cfg:
        mc_cfg['samples'] = raw_cfg['mc_samples']
    if 'mc_seed' in raw_cfg and 'seed' not in mc_cfg:
        mc_cfg['seed'] = raw_cfg['mc_seed']
    if 'seed' not in mc_cfg:
        mc_cfg['seed'] = seed
    if 'samples' not in mc_cfg:
        mc_cfg['samples'] = 5000
    cfg['mc'] = mc_cfg

    REQUIRED = ['alpha_dB_per_km','L0_km','eta_s','eta_w','eta_r','eta_det','V0','T2_ms','N_modes','connector_loss_dB','n_connectors','splice_loss_dB','n_splices','filter_loss_dB','c_fiber_mps','geometry','mc']
    messages=[]; table=[]; ok=True
    for key in REQUIRED:
        val = cfg.get(key)
        unit = SCHEMA.get(key,{}).get('unit','')
        lo = SCHEMA.get(key,{}).get('min'); hi = SCHEMA.get(key,{}).get('max')
        if val is None:
            ok=False; messages.append(f'Missing required key: {key}')
        else:
            if key!='mc' and isinstance(val,(int,float)):
                if lo is not None and val < lo: ok=False; messages.append(f'{key}={val} < {lo}')
                if hi is not None and val > hi: ok=False; messages.append(f'{key}={val} > {hi}')
        table.append((key,val,unit,lo,hi))

    # Derive n_swaps if needed
    if cfg.get('total_distance_km') is not None and cfg.get('n_swaps') is None and cfg.get('L0_km'):
        try:
            cfg['n_swaps'] = max(0,int(round(cfg['total_distance_km']/cfg['L0_km']) - 1))
            messages.append(f'Derived n_swaps={cfg['n_swaps']}')
        except Exception:
            ok=False; messages.append('Failed deriving n_swaps')
    elif cfg.get('n_swaps') is None:
        cfg['n_swaps'] = 0

    print('[Flow 1] Schema table (key | value | unit | min..max):')
    for key,val,unit,lo,hi in table:
        print(f'  {key} | {val} | {unit} | [{lo},{hi}]')
    if messages:
        print('[Flow 1] Notes:'); [print(' ',m) for m in messages]

    FLOW_STATE['config_ok']=ok
    hash_basis={k:cfg[k] for k in cfg if k!='mc'}; hash_basis['mc_samples']=cfg['mc']['samples']
    FLOW_STATE['schema_hash']=_hashlib.sha256(_json.dumps(hash_basis,sort_keys=True).encode()).hexdigest()[:16]
    FLOW_STATE['preset_id']=cfg.get('preset_id')

    if not ok:
        print('[Flow 0–3] Aborting after config validation failure.')
    else:
        # Marginals (stochastic subset only)
        # NOTE: c_fiber_mps is intentionally treated as a constant (physical fiber speed)
        # and therefore NOT included among stochastic marginals to avoid PIT distortion.
        if 'WORKING_MARGINALS' not in globals() or not WORKING_MARGINALS:
            WORKING_MARGINALS={
                'eta_s': {'family':'beta','params':{'a':70,'b':30},'unit':'frac'},
                'eta_det': {'family':'beta','params':{'a':75,'b':25},'unit':'frac'},
                'alpha_dB_per_km': {'family':'lognormal','params':{'mu_log':_np.log(cfg['alpha_dB_per_km']), 'sigma_log':0.05},'unit':'dB/km'},
                'L0_km': {'family':'uniform','params':{'low':cfg['L0_km']*0.8,'high':cfg['L0_km']*1.2},'unit':'km'},
                'V0': {'family':'beta','params':{'a':90,'b':10},'unit':'vis'},
                'T2_ms': {'family':'lognormal','params':{'mu_log':_np.log(max(cfg['T2_ms'],1e-9)),'sigma_log':0.1},'unit':'ms'},
                'N_modes': {'family':'uniform','params':{'low':cfg['N_modes']*0.9,'high':cfg['N_modes']*1.1},'unit':'count'},
                'n_swaps': {'family':'uniform','params':{'low':max(cfg.get('n_swaps',0)-1,0),'high':cfg.get('n_swaps',0)+1},'unit':'count'},
                'kappa_power': {'family':'uniform','params':{'low':1.0,'high':1.2},'unit':'power'},
            }
        marg=WORKING_MARGINALS; bound={}
        def _bind_margin(key,spec):
            fam=spec['family'].lower(); params=spec.get('params',{}); unit=spec.get('unit','')
            if fam=='beta': a=float(params['a']); b=float(params['b']); cdf=lambda x:_beta.cdf(x,a,b); ppf=lambda u:_beta.ppf(u,a,b)
            elif fam=='lognormal': mu=float(params['mu_log']); sigma=float(params['sigma_log']); s=sigma; scale=_np.exp(mu); cdf=lambda x:_lognorm.cdf(x,s,scale=scale); ppf=lambda u:_lognorm.ppf(u,s,scale=scale)
            elif fam=='uniform': low=float(params['low']); high=float(params['high']); cdf=lambda x:_uniform.cdf(x,loc=low,scale=high-low); ppf=lambda u:_uniform.ppf(u,loc=low,scale=high-low)
            elif fam in ('truncnorm','truncated_normal'): lo=float(params['low']); hi=float(params['high']); loc=float(params.get('loc',0.0)); scale=float(params.get('scale',1.0)); a=(lo-loc)/scale; b=(hi-loc)/scale; cdf=lambda x:_truncnorm.cdf(x,a,b,loc=loc,scale=scale); ppf=lambda u:_truncnorm.ppf(u,a,b,loc=loc,scale=scale)
            else: raise ValueError(f'Unsupported margin family {fam}')
            return {'cdf':cdf,'ppf':ppf,'unit':unit,'family':spec['family'],'params':params}
        try:
            for k,s in marg.items(): bound[k]=_bind_margin(k,s)
            FLOW_STATE['marginals_bound']=bound; FLOW_STATE['marginals_ready']=True
            print('[Flow 2] Marginals bound:');
            for k,b in bound.items(): print(f"  {k}: {b['family']} params={b['params']} unit={b['unit']}")
        except Exception as e:
            FLOW_STATE['marginals_ready']=False; print('[Flow 2] Marginal binding failed:',e)

        # Copula assembly
        tau_pairs = raw_cfg.get('copula',{}).get('kendall_tau_pairs', [])
        FLOW_STATE['tau_pairs']=[(a,b,float(t)) for a,b,t in tau_pairs]
        var_order=list(bound.keys()); FLOW_STATE['var_order']=var_order
        if FLOW_STATE.get('marginals_ready') and tau_pairs:
            idx={k:i for i,k in enumerate(var_order)}; tau_mat=_np.zeros((len(var_order),len(var_order)))
            for a,b,t in tau_pairs:
                if a in idx and b in idx: ia,ib=idx[a],idx[b]; tau_mat[ia,ib]=tau_mat[ib,ia]=float(t)
            rho_psd,corr_norm=tau_to_rho_matrix(tau_mat)
            FLOW_STATE['Sigma_psd']=rho_psd; FLOW_STATE['psd_correction_norm']=float(corr_norm)
            try:
                L=robust_cholesky(rho_psd) if 'robust_cholesky' in globals() else _np.linalg.cholesky(rho_psd+1e-12*_np.eye(len(var_order)))
                FLOW_STATE['chol_L']=L; FLOW_STATE['copula_ready']=True
            except Exception as e:
                FLOW_STATE['copula_ready']=False; print('[Flow 3] Cholesky failed:',e)
            eig=_np.linalg.eigvalsh(rho_psd)
            print('[Flow 3] Copula:'); print(f'  pair_count={len(tau_pairs)} avg|τ|={(_np.mean(_np.abs([t for _,_,t in tau_pairs])) if tau_pairs else 0.0):.3f}')
            print(f'  PSD correction norm={float(corr_norm):.3e} eig[min,max]=[{eig.min():.3e},{eig.max():.3e}]')
            print('  Σ_psd[0:3,0:3]='); print(_np.array2string(rho_psd[:3,:3],precision=3,suppress_small=True))
        else:
            # If no tau_pairs were provided, fall back to independent Gaussian copula (identity correlation).
            n = len(var_order)
            FLOW_STATE['Sigma_psd'] = _np.eye(n)
            FLOW_STATE['psd_correction_norm'] = 0.0
            FLOW_STATE['chol_L'] = _np.eye(n)
            FLOW_STATE['copula_ready'] = True
            print('[Flow 3] No tau_pairs provided — Gaussian copula fallback to independence (Σ = I).')
except Exception as e:
    print('Flow Step 0–3 refactor error:', e)


[Flow 0] seed=123456789 device=cpu dtype=float64
[Flow 1] Schema table (key | value | unit | min..max):
  alpha_dB_per_km | 0.2 | dB/km | [0.0,None]
  L0_km | 25.0 | km | [0.0,None]
  eta_s | 0.6 | frac | [0.0,1.0]
  eta_w | 0.7 | frac | [0.0,1.0]
  eta_r | 0.7 | frac | [0.0,1.0]
  eta_det | 0.9 | frac | [0.0,1.0]
  V0 | 0.9 | vis | [0.0,1.0]
  T2_ms | 150.0 | ms | [0.01,None]
  N_modes | 50.0 | count | [1.0,None]
  connector_loss_dB | 0.5 | dB | [0.0,None]
  n_connectors | 2 | count | [0,None]
  splice_loss_dB | 0.1 | dB | [0.0,None]
  n_splices | 0 | count | [0,None]
  filter_loss_dB | 0.0 | dB | [0.0,None]
  c_fiber_mps | 200000000.0 | m/s | [1000000.0,None]
  geometry | central_bsm | enum | [None,None]
  mc | {'seed': 123456789, 'samples': 5000} |  | [None,None]
[Flow 2] Marginal binding failed: 'family'
[Flow 3] No tau_pairs provided — Gaussian copula fallback to independence (Σ = I).


## Bell / CHSH inequality
Goal: distinguish classical local-hidden-variable (LHV) correlations from quantum entanglement and quantify nonlocality used in E91 and DI-QKD (Bell 1964 https://doi.org/10.1103/PhysicsPhysiqueFizika.1.195; CHSH 1969 https://doi.org/10.1103/PhysRevLett.23.880; Tsirelson bound 1980 https://doi.org/10.1007/BF00417500; Ekert E91 1991 (journal: Phys. Rev. Lett. 67, 661)).

### Setup
Two parties (Alice, Bob) share a bipartite state $\rho_{AB}$ (ideal singlet $|\Psi^-\rangle = (|01\rangle - |10\rangle)/\sqrt{2}$). They choose measurement settings $a\in\{a_0,a_1\}$, $b\in\{b_0,b_1\}$ corresponding to spin projection directions (Pauli observables $\vec n\cdot\vec \sigma$). Outcomes $A,B\in\{-1,+1\}$.

### Correlation functions
$$
E(a_i,b_j)=\sum_{A,B\in\{-1,+1\}} AB\, P(A,B\mid a_i,b_j).
$$
For a pure state and projective measurements:
$$
E(a_i,b_j)=\langle\Psi^-| (\vec n_{a_i}\cdot\vec \sigma) \otimes (\vec n_{b_j}\cdot\vec \sigma) |\Psi^-\rangle = -\vec n_{a_i}\cdot\vec n_{b_j}.
$$
Note: the overall sign depends on the chosen Bell state.

### CHSH quantity
$$
S = E(a_0,b_0)+E(a_0,b_1)+E(a_1,b_0)-E(a_1,b_1).
$$

### Classical (LHV) bound
Any LHV model satisfies $|S| \le 2$ (CHSH 1969 https://doi.org/10.1103/PhysRevLett.23.880).

### Quantum Tsirelson bound
Quantum mechanics allows $|S| \le 2\sqrt{2}$ (Tsirelson 1980 https://doi.org/10.1007/BF00417500). Optimal choice (e.g.): angles in a plane at $0,\pi/2$ for Alice; $\pi/4, -\pi/4$ for Bob yield $S=2\sqrt{2}$.

### Violation and security
Observed $S>2$ implies genuine entanglement and rules out classical shared randomness explanation. In Ekert E91 (1991 Phys. Rev. Lett. 67, 661) fraction of rounds used to estimate $S$; remaining rounds (with a chosen basis subset) form raw key. Larger violation tightens bounds on Eve’s accessible information.

### Finite-size and loopholes
- Detection loophole: need high overall efficiency to avoid biasing sample.
- Locality loophole: space-like separation of setting choices and measurements.
- Freedom-of-choice: random setting generation independent of source.
DI-QKD demands all major loopholes closed.

### Relation to secret key rate (sketch)
Given observed $S$ and QBER $q$, security proofs bound Eve’s Holevo information $\chi_{BE}$, leading to key rate lower bounds $K \gtrsim 1 - h(q) - f(S)$ where $h$ is binary entropy and $f$ decreases with larger $S$ (exact form protocol-dependent).

### Other inequalities
- Mermin/GHZ for multipartite nonlocality.
- Eberhard form optimized for low detection efficiency.

### How to run the next cell
- Run the next code cell to execute sampling/diagnostics or subsequent computations that build on these concepts.
- If you changed any parameters above, re-run the next cell to apply updates.


In [11]:
# Flow Step 4–5: Joint sampling (Gaussian copula) → Diagnostics
# Order-independent: auto-ensure marginals and copula from CONFIG if not present.

try:
    import numpy as _np
    from scipy.stats import norm as _norm
    from scipy.stats import kstest as _kstest
    from scipy.stats import kendalltau as _kendalltau

    if 'FLOW_STATE' not in globals():
        FLOW_STATE = {}

    def _ensure_marginals_and_copula():
        # Use the converter utility to create WORKING_MARGINALS_LIST and VAR_ORDER
        if not FLOW_STATE.get('WORKING_MARGINALS_LIST'):
            convert_to_working_marginals(defaults=globals().get('DEFAULT_MARGINALS', {}), working=globals().get('WORKING_MARGINALS', {}))
        # Expose a backward-compatible bound dict mapping key -> {ppf,cdf,unit,family,params}
        wlist = globals().get('WORKING_MARGINALS_LIST', [])
        bound = {}
        for m in wlist:
            bound[m['key']] = {'ppf': m['ppf'], 'cdf': m['cdf'], 'unit': m.get('unit',''), 'family': m.get('family',''), 'params': m.get('params',{})}
        FLOW_STATE['marginals_bound'] = bound
        FLOW_STATE['marginals_ready'] = True
        FLOW_STATE['var_order'] = globals().get('VAR_ORDER', [m['key'] for m in wlist])

        # Copula assembly using build_copula utility
        raw_cfg = globals().get('CONFIG') or {}
        cop_cfg = raw_cfg.get('copula', {}) or {}
        tau_pairs = cop_cfg.get('kendall_tau_pairs', [])
        FLOW_STATE['tau_pairs'] = [(a,b,float(t)) for a,b,t in tau_pairs]
        var_order = FLOW_STATE['var_order']
        if tau_pairs:
            cop = build_copula(var_order, tau_pairs)
            FLOW_STATE['Sigma_psd'] = cop['Sigma_psd']
            FLOW_STATE['psd_correction_norm'] = cop['diag']['psd_correction_norm']
            FLOW_STATE['chol_L'] = cop['L']
            FLOW_STATE['copula_ready'] = True
            FLOW_STATE['copula_diag'] = cop['diag']
        else:
            # Identity fallback
            n = len(var_order)
            FLOW_STATE['Sigma_psd'] = _np.eye(n)
            FLOW_STATE['psd_correction_norm'] = 0.0
            FLOW_STATE['chol_L'] = _np.eye(n)
            FLOW_STATE['copula_ready'] = True

    # Ensure readiness regardless of previous execution order
    if not (FLOW_STATE.get('copula_ready', False) and FLOW_STATE.get('marginals_ready', False)):
        _ensure_marginals_and_copula()

    state_ok = FLOW_STATE.get('copula_ready', False) and FLOW_STATE.get('marginals_ready', False)
    if not state_ok:
        print('[Flow 4–5] Skipped: copula/marginals not ready.')
    else:
        # thresholds from CONFIG
        diag_cfg = (globals().get('CONFIG') or {}).get('diag', {}) or {}
        ks_alpha = float(diag_cfg.get('ks_alpha', 0.05))
        tau_tol = float(diag_cfg.get('tau_tol', 0.05))
        min_N_for_tau = int(diag_cfg.get('min_N_for_tau', 5000))
        force_proceed = bool(diag_cfg.get('force_proceed', False))
        # allow diagnostics override via CONFIG.run.allow_diagnostics_override (default False)
        allow_override = bool((globals().get('CONFIG') or {}).get('run', {}).get('allow_diagnostics_override', False))

        N = int(FLOW_STATE.get('mc_N', 0) or (globals().get('CONFIG') or {}).get('mc',{}).get('samples', 0) or 0)
        if N <= 0:
            N = 10000
            print('[Flow 4] Using default sample size N=10000')
        seed = int(FLOW_STATE.get('seed', globals().get('GLOBAL_SEED', 123456789)))
        rng = _np.random.default_rng(seed)
        L = FLOW_STATE['chol_L']
        var_order = FLOW_STATE['var_order']
        bound = FLOW_STATE['marginals_bound']

        # Z -> Zc -> U
        Z = rng.standard_normal(size=(N, len(var_order)))
        Zc = Z @ L.T
        U = _norm.cdf(Zc)
        U = _np.clip(U, 1e-12, 1-1e-12)

        # Map to samples via PPF
        X = {}
        for j, key in enumerate(var_order):
            ppf = bound[key]['ppf']
            X[key] = _np.asarray(ppf(U[:, j]))
        FLOW_STATE['samples_X'] = X
        FLOW_STATE['U'] = U
        FLOW_STATE['mc_N'] = N

        # Show 5-row head
        print('[Flow 4] Samples head:')
        head_rows = min(5, N)
        for key in var_order:
            unit = bound[key].get('unit','')
            print(f'  {key} [{unit}]:', _np.array2string(X[key][:head_rows], precision=4))

        # Validation: print min/median/max for each sampled variable to catch collapsed variances
        print('[Flow 4] Sample ranges (min | median | max):')
        for key in var_order:
            try:
                arr = _np.asarray(X[key])
                mn = float(_np.nanmin(arr))
                md = float(_np.nanmedian(arr))
                mx = float(_np.nanmax(arr))
                print(f'  {key}: {mn:.3e} | {md:.3e} | {mx:.3e}')
            except Exception:
                print(f'  {key}: (non-numeric or missing)')

        # Diagnostics
        print('[Flow 5] Diagnostics: PIT and Kendall τ checks')
        pit_results = {}
        ks_fail = []
        for j, key in enumerate(var_order):
            # Skip constant variables in PIT
            if bound.get(key,{}).get('family') in ('const','constant') or _np.std(X[key]) < 1e-15:
                continue
            cdf = bound[key]['cdf']
            U_emp = _np.clip(cdf(X[key]), 1e-12, 1-1e-12)
            stat, pval = _kstest(U_emp, 'uniform')
            pit_results[key] = {'ks_stat': float(stat), 'p_value': float(pval)}
            if pval < ks_alpha:
                ks_fail.append(key)

        tau_pairs = FLOW_STATE.get('tau_pairs', [])
        tau_results = []
        tau_fail = []
        for (i_key, j_key, tau_target) in tau_pairs:
            if i_key in X and j_key in X:
                tau_emp, _ = _kendalltau(X[i_key], X[j_key])
                delta = abs(float(tau_emp) - float(tau_target))
                tau_results.append((i_key, j_key, float(tau_target), float(tau_emp), float(delta)))
                if delta > tau_tol and N >= min_N_for_tau:
                    tau_fail.append((i_key, j_key))

        print('  PIT results (KS p-values):')
        for k,v in pit_results.items():
            print(f"    {k}: p={v['p_value']:.3f}")
        if tau_results:
            print('  Kendall τ targets vs empirical:')
            for i_key,j_key,tau_t,tau_e,delta in tau_results:
                print(f'    ({i_key},{j_key}): target={tau_t:+.3f}, emp={tau_e:+.3f}, |Δ|={delta:.3f}')

        diagnostics_ok = (len(ks_fail)==0) and (len(tau_fail)==0)
        # legacy/diag force flag preserved
        if not diagnostics_ok and force_proceed:
            print('[Flow 5] Diagnostics failed, but force_proceed=True; continuing.')
            diagnostics_ok = True
        # New CONFIG-driven allow override
        if not diagnostics_ok and allow_override:
            print('[Flow 5] Diagnostics failed, but CONFIG.run.allow_diagnostics_override=True; running physics provisionally.')
            FLOW_STATE['diagnostics_provisional'] = True
            diagnostics_ok = True
        FLOW_STATE['diagnostics_ok'] = diagnostics_ok
        if not diagnostics_ok:
            print('[Flow 5] Diagnostics failed; skipping physics.')
        else:
            print('[Flow 5] Diagnostics OK.')
except Exception as e:
    print('Flow Step 4–5 error:', e)


Flow Step 4–5 error: name 'convert_to_working_marginals' is not defined


In [12]:
# 8) Backend Detection (CUDA/DirectML/CPU) and Guidance Printers

import subprocess, shutil, platform

def detect_hardware():
    hw = {
        'nvidia_smi': False,
        'nvidia_gpus': [],
        'torch': HAS_TORCH,
        'cuda_available': False,
        'dml': False,
    }
    try:
        res = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True, check=False)
        if res.returncode == 0 and res.stdout.strip():
            hw['nvidia_smi'] = True
            hw['nvidia_gpus'] = [l.strip() for l in res.stdout.splitlines() if l.strip()]
    except Exception:
        pass
    if HAS_TORCH:
        try:
            hw['cuda_available'] = bool(torch.cuda.is_available())
        except Exception:
            hw['cuda_available'] = False
    try:
        import torch_directml  # noqa: F401
        hw['dml'] = True
    except Exception:
        hw['dml'] = False
    return hw


def print_conda_cuda_instructions(env_name='qnet-cuda'):
    print('\nConda steps for CUDA-enabled env:')
    print(f'  conda create -n {env_name} python=3.11 -y')
    print(f'  conda activate {env_name}')
    print('  conda install -y -c pytorch -c nvidia pytorch torchvision torchaudio pytorch-cuda=12.1')
    print('Start Jupyter from that env and re-open this notebook.')


def print_dml_venv_instructions(venv_path='venvs/qnet-dml'):
    print('\nWindows venv steps for DirectML:')
    print(f'  python -m venv {venv_path}')
    print(f'  {venv_path}\\Scripts\\activate')
    print('  python -m pip install --upgrade pip setuptools wheel')
    print('  python -m pip install torch-directml')
    print('Start Jupyter from that venv and re-open this notebook.')

print('Hardware detection helpers ready (CPU default).')

Hardware detection helpers ready (CPU default).


In [13]:
# 9) Backend Selection UI (ipywidgets)
# Dropdowns for backend and device; Apply button sets NOTEBOOK_BACKEND/SELECTED_DEVICE

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    _BE_widgets = globals().get('_backend_select_widgets')
    if _BE_widgets:
        for w in _BE_widgets:
            try: w.close()
            except Exception: pass
    _BE_widgets = []

    hw = detect_hardware()
    options = [('CPU','cpu')]
    if hw.get('cuda_available') or hw.get('nvidia_gpus'):
        options.insert(0, ('CUDA (GPU)','cuda'))
    if hw.get('dml'):
        options.insert(1, ('DirectML (GPU)','dml'))

    backend_dd = widgets.Dropdown(options=options, value='cpu', description='Backend:')
    device_dd = widgets.Dropdown(options=[('CPU','cpu')], description='Device:')
    status = widgets.HTML(value='')
    apply_btn = widgets.Button(description='Apply', button_style='success')

    def _on_backend_change(ch):
        if ch['name'] == 'value':
            if ch['new'] == 'cuda' and hw.get('nvidia_gpus'):
                device_dd.options = [(f'CUDA {i}: {name}', f'cuda:{i}') for i, name in enumerate(hw['nvidia_gpus'])] or [('CUDA:0','cuda:0')]
                device_dd.value = device_dd.options[0][1]
            elif ch['new'] == 'dml':
                device_dd.options = [('DirectML','dml')]
                device_dd.value = 'dml'
            else:
                device_dd.options = [('CPU','cpu')]
                device_dd.value = 'cpu'
            status.value = ''

    backend_dd.observe(_on_backend_change, names='value')

    def _on_apply(_):
        global NOTEBOOK_BACKEND, SELECTED_DEVICE
        NOTEBOOK_BACKEND = backend_dd.value
        SELECTED_DEVICE = device_dd.value
        status.value = f"<span style='color:#5cb85c'>Selected: <b>{NOTEBOOK_BACKEND}</b> on <b>{SELECTED_DEVICE}</b></span>"

    apply_btn.on_click(_on_apply)

    panel = widgets.VBox([
        widgets.HTML("<b>Backend selection</b>"),
        widgets.HBox([backend_dd, device_dd, apply_btn]),
        widgets.HTML("<div class='small-hint'>CPU is recommended unless you have a configured GPU env.</div>"),
        status
    ])
    display(panel)
    _BE_widgets.extend([backend_dd, device_dd, apply_btn, status, panel])
    globals()['_backend_select_widgets'] = _BE_widgets
except Exception as e:
    print('Backend UI unavailable:', e)


# Single-run engine and visualization helpers

This block lets you execute a single full Monte Carlo run and inspect results.

### What happens when you click Run
1. Configuration load: tries `config/marginal_config.json` or a CSV catalog; if missing, falls back to the current in-memory working marginals.
2. Sampling: draws `N` samples per marginal using the inline sampler (`sample_marginal_inline`).
3. Metrics: passes sampled arrays to the physics kernels to compute throughput R and related metrics.
4. Visualization: shows histograms for each input and metric; wide positive ranges automatically get a log-x axis.
5. Persistence: saves `<timestamp>_summary.json` (metadata) and `<timestamp>_samples.npz` (arrays).

### UI elements explained (next cell)
- Backend: choose 'CPU'/'NumPy' for local inline path; 'CUDA'/'DirectML' only if a device runner is implemented.
- Samples: target number of Monte Carlo samples (blank uses a default). Clamped to [1e4, 1e7].
- Run: triggers the end-to-end pipeline.

### Recommended workflow
1. Configure marginals first in the Marginal Configuration UI.
2. Optionally adjust physics parameters (loss, efficiencies) above.
3. Set Samples (e.g. 200000 for smoother CDF).
4. Click Run; inspect plots and summary; adjust and re-run as needed.

### Output artifacts
- Summary JSON: run name, backend, sample count, metric keys.
- NPZ: sampled marginals + metric arrays for later analysis.

### Troubleshooting
- Missing marginals: ensure you saved a config or built marginals inline.
- Memory pressure: lower Samples (histograms subsample when large).
- Degenerate distributions: revisit configuration (constant or all-zero can suppress variability in R).


### How to use the next cell

- Run the following code cell to execute this section’s computations or UI.
- Configure any inputs above first; then press ▶ or Shift+Enter.
- Review the output; re-run after any configuration change.

In [14]:
# 12) Single-Run Inline/Device Runner and Visual Diagnostics
# Loads config (JSON/CSV), chooses inline vs device path, saves samples+metrics, visualizes distributions.

import json as _json, os as _os
from pathlib import Path
import time
try:
    import pandas as _pd
    _HAS_PANDAS = True
except Exception:
    _HAS_PANDAS = False

try:
    import matplotlib.pyplot as _plt
    _HAS_MPL = True
except Exception:
    _HAS_MPL = False

_DEF_MIN_SAMPLES = 10000
_DEF_MAX_SAMPLES = int(1e7)


def _brief_stats(x):
    return dict(min=float(_np.min(x)), p5=float(_np.percentile(x,5)), median=float(_np.median(x)), p95=float(_np.percentile(x,95)), max=float(_np.max(x)), mean=float(_np.mean(x)))


def _viz_arrays(named_arrays, title_prefix, max_points=50000, cols=3):
    if not _HAS_MPL:
        print('Matplotlib not available for arrays visualization.')
        return
    items = [(k,v) for k,v in named_arrays.items() if isinstance(v, _np.ndarray)]
    if not items:
        print('No arrays to visualize.')
        return
    N = len(items[0][1])
    if N > max_points:
        rng = _np.random.default_rng(123)
        idx = rng.choice(N, size=max_points, replace=False)
    else:
        idx = slice(None)
    rows = (len(items)+cols-1)//cols
    fig, axes = _plt.subplots(rows, cols, figsize=(cols*5.0, rows*3.5))
    axes = _np.atleast_1d(axes).reshape(rows, cols)
    for i,(k,a) in enumerate(items):
        a = a[idx]
        r = i//cols; c = i%cols
        ax = axes[r,c]
        bins = 90 if len(a) > 4000 else max(30, int(len(a)//70))
        ax.hist(a, bins=bins, color='#4C72B0', alpha=0.85)
        st = _brief_stats(a)
        ax.set_title(f"{k}\nmin={st['min']:.3e} med={st['median']:.3e} max={st['max']:.3e}")
        try:
            if _np.all(a>0) and _np.max(a)/max(1e-30,_np.min(a)) > 1e3:
                ax.set_xscale('log')
        except Exception:
            pass
        ax.grid(True, alpha=0.3)
    for j in range(i+1, rows*cols):
        axes[j//cols, j%cols].axis('off')
    fig.suptitle(f'{title_prefix} histograms')
    fig.tight_layout(); _plt.show()


def _viz_metrics(metrics, max_points=50000, cols=3):
    if not _HAS_MPL:
        print('Matplotlib not available for metrics visualization.')
        return
    arr_items = [(k,v) for k,v in metrics.items() if isinstance(v, _np.ndarray)]
    if not arr_items:
        print('No metric arrays to visualize.')
        return
    N = len(arr_items[0][1])
    if N > max_points:
        rng = _np.random.default_rng(321)
        idx = rng.choice(N, size=max_points, replace=False)
    else:
        idx = slice(None)
    rows = (len(arr_items)+cols-1)//cols
    fig, axes = _plt.subplots(rows, cols, figsize=(cols*5.0, rows*3.5))
    axes = _np.atleast_1d(axes).reshape(rows, cols)
    for i,(k,a) in enumerate(arr_items):
        a = a[idx]
        r=i//cols; c=i%cols
        ax = axes[r,c]
        bins = 90 if len(a)>4000 else max(30,int(len(a)//70))
        ax.hist(a, bins=bins, color='#55A868', alpha=0.85)
        st = _brief_stats(a)
        ax.set_title(f"{k}\nmin={st['min']:.3e} med={st['median']:.3e} max={st['max']:.3e}")
        try:
            if _np.all(a>0) and _np.max(a)/max(1e-30,_np.min(a))>1e3:
                ax.set_xscale('log')
        except Exception:
            pass
        ax.grid(True, alpha=0.3)
    for j in range(i+1, rows*cols):
        axes[j//cols, j%cols].axis('off')
    fig.suptitle('Metrics histograms'); fig.tight_layout(); _plt.show()
    if 'R' in metrics and isinstance(metrics['R'], _np.ndarray):
        r = metrics['R'][idx]
        r_sorted = _np.sort(r); y = _np.linspace(0,1,len(r_sorted), endpoint=False)
        _plt.figure(figsize=(6,4))
        _plt.plot(r_sorted, y, lw=2)
        _plt.xlabel('R'); _plt.ylabel('CDF'); _plt.title('CDF of R'); _plt.grid(True, alpha=0.3)
        try:
            if _np.all(r_sorted>0) and r_sorted[-1]/max(1e-30,r_sorted[0])>1e3:
                _plt.xscale('log')
        except Exception:
            pass
        _plt.show()
    if 'R' in metrics and 'R_legacy' in metrics:
        R = metrics['R'][idx]; Rl = metrics['R_legacy'][idx]
        _plt.figure(figsize=(5,5))
        _plt.scatter(Rl, R, s=5, alpha=0.3)
        lim = [min(_np.min(Rl), _np.min(R)), max(_np.max(Rl), _np.max(R))]
        lim[0] = max(lim[0], 1e-30); lim[1] = max(lim[1], lim[0]*1.1)
        _plt.plot(lim, lim, 'r--', lw=1)
        try:
            _plt.xscale('log'); _plt.yscale('log')
        except Exception:
            pass
        _plt.xlabel('R_legacy'); _plt.ylabel('R (new)'); _plt.title('R new vs R legacy')
        _plt.grid(True, alpha=0.3)
        _plt.show()


def _load_config_candidates():
    candidates = [CONFIG_DIR/'marginal_config.json', CONFIG_DIR/'marginal_catalog.csv', Path('marginal_config.json'), Path('marginal_catalog.csv')]
    seen = []
    for p in candidates:
        p = Path(p)
        if p in seen: continue
        seen.append(p)
        if p.exists():
            yield p


def _load_config():
    cfg_df = None; cfg_raw = None
    for p in _load_config_candidates():
        try:
            if p.suffix.lower()=='.csv':
                if not _HAS_PANDAS: continue
                df = _pd.read_csv(p)
                if df is not None and not df.empty:
                    cfg_df = df; break
            else:
                raw = _json.load(open(p,'r',encoding='utf-8'))
                if isinstance(raw, dict) and 'marginals' in raw:
                    rows = []
                    for m in raw['marginals']:
                        variable = m.get('variable') or m.get('name') or m.get('key')
                        dist = m.get('dist') or m.get('family')
                        params = m.get('params', {})
                        rows.append({'variable': variable, 'dist': dist, 'params': params})
                    cfg_df = _pd.DataFrame(rows) if _HAS_PANDAS else None
                    cfg_raw = rows if not _HAS_PANDAS else None
                    break
                elif isinstance(raw, list):
                    # list of marginal specs
                    rows = []
                    for m in raw:
                        variable = m.get('variable') or m.get('name') or m.get('key') or m.get('key')
                        dist = m.get('dist') or m.get('family')
                        params = m.get('params', {})
                        rows.append({'variable':variable,'dist':dist,'params':params})
                    cfg_df = _pd.DataFrame(rows) if _HAS_PANDAS else None
                    cfg_raw = rows if not _HAS_PANDAS else None
                    break
        except Exception as e:
            print('Failed to parse', p, '->', e)
    if cfg_df is None and cfg_raw is None and 'WORKING_MARGINALS' in globals():
        rows = []
        for k,v in WORKING_MARGINALS.items():
            rows.append({'variable':k,'dist': v.get('dist') or v.get('family'), 'params': v.get('params',{})})
        cfg_df = _pd.DataFrame(rows) if _HAS_PANDAS else None
        cfg_raw = rows if not _HAS_PANDAS else None
    return cfg_df, cfg_raw


def run_once(backend: str, samples: int):
    timestamp = time.strftime('%Y%m%dT%H%M%S')
    run_name = f'run_{timestamp}'
    # Ensure RESULTS_DIR exists
    global RESULTS_DIR
    if 'RESULTS_DIR' not in globals() or RESULTS_DIR is None:
        RESULTS_DIR = Path('results')
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    summary_path = RESULTS_DIR / f'{run_name}_summary.json'
    samples_path = RESULTS_DIR / f'{run_name}_samples.npz'

    cfg_df, cfg_raw = _load_config()
    sample_fn = globals().get('sample_marginal_inline')
    prefer_inline = backend in ('cpu','numpy') and callable(sample_fn)

    # ensure RNG
    seed = int(globals().get('GLOBAL_SEED', 123456789))
    RNG_local = globals().get('RNG') or _np.random.default_rng(seed)

    if (cfg_df is not None or cfg_raw is not None) and prefer_inline:
        sampled = {}
        rows = cfg_df.to_dict(orient='records') if cfg_df is not None else cfg_raw
        for r in rows:
            key = r.get('variable') or r.get('key')
            # normalize row for sample function
            dist_name = r.get('dist') or r.get('family')
            params = r.get('params', {}) or {}
            row_spec = {'dist': dist_name, 'params': params}
            if not key:
                # if variable/key missing, try to infer from params (empirical) or skip
                continue
            sampled[key] = sample_fn(row_spec, samples, rng=RNG_local)
        _viz_arrays(sampled, 'Sampled inputs')
        # If metrics function expects dict of numpy arrays with particular keys, pass sampled as-is
        metrics = globals().get('_compute_metrics_cpu')(sampled)
    elif (cfg_df is not None) and callable(globals().get('run_on_device')):
        # device path (external runner) expects a dataframe
        sampled, metrics, _ = run_on_device(cfg_df, sampleN=samples, device_str=backend)
        _viz_arrays(sampled, 'Sampled inputs')
    else:
        print('No usable config. Create marginals first.')
        return

    _viz_metrics(metrics)
    summary = {'run_name': run_name, 'backend': backend, 'sampleN': samples, 'metrics_keys': list(metrics.keys())}
    with open(summary_path,'w',encoding='utf-8') as fh:
        _json.dump(summary, fh, indent=2)
    try:
        # Save numeric arrays only (sampled may include non-array entries)
        tosave = {k:v for k,v in sampled.items() if isinstance(v, _np.ndarray)}
        tosave_metrics = {k:v for k,v in metrics.items() if isinstance(v,_np.ndarray)}
        _np.savez_compressed(samples_path, **tosave, **tosave_metrics)
    except Exception as e:
        print('Warning: could not save samples ->', e)
    print('Run finished. Summary:', summary_path, 'Samples+metrics:', samples_path)

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    backend_dd = widgets.Dropdown(options=[('CPU','cpu'),('NumPy','numpy'),('CUDA','cuda'),('DirectML','dml')], value='cpu', description='Backend:')
    sample_txt = widgets.Text(value='', placeholder=f'>={_DEF_MIN_SAMPLES}', description='Samples:')
    run_btn = widgets.Button(description='Run', button_style='success')
    out = widgets.Output()

    def _parse_samples(s):
        s = (s or '').strip()
        if not s: return _DEF_MIN_SAMPLES
        try: n = int(s)
        except Exception: return _DEF_MIN_SAMPLES
        if n < _DEF_MIN_SAMPLES: n = _DEF_MIN_SAMPLES
        if n > _DEF_MAX_SAMPLES: n = _DEF_MAX_SAMPLES
        return n

    def _on_run(_):
        with out:
            clear_output(wait=True)
            backend = backend_dd.value
            samples = _parse_samples(sample_txt.value)
            print(f'Backend={backend} Samples={samples}')
            run_once(backend, samples)

    run_btn.on_click(_on_run)
    display(widgets.VBox([widgets.HBox([backend_dd, sample_txt, run_btn]), out]))
except Exception as e:
    print('Run UI unavailable; fallback to text prompts.', e)


# Results visualization UI (R histograms & ECDF)

This block provides an interactive UI for exploring previously saved runs.

### Controls (next cell)
- Samples: choose a saved `*_samples.npz` file.
- Summary: choose a corresponding `*_summary.json` (optional; used for context).
- Reconstruct R if missing: if the NPZ lacks `R`, tries to recompute it from available `m_*` arrays via physics kernels.
- KDE overlay: overlays a smooth density estimate on the histogram (qualitative visual aid).
- Log X: plot the histogram with a log x-axis when `R>0`.
- Refresh list: rescan the results folder to update the dropdowns.
- Plot: render histogram and ECDF with the chosen options.

### How to use
1) Click Refresh list to populate file dropdowns after new runs.
2) Pick a Samples NPZ (and Summary JSON if helpful).
3) Toggle KDE/Log X/Reconstruct options as needed.
4) Click Plot to render.

Mathematical notes:
- Empirical CDF: $\hat F_n(x)=\tfrac{1}{n}|\{R_i \le x\}|$ shows quantiles at a glance.
- KDE: $\hat f(x)=\tfrac{1}{nh}\sum K\big((x-R_i)/h\big)$ with Gaussian $K$; used only for visualization here.


### How to use the next cell

- Execute the next code cell to run the associated UI or computation.
- Set any widgets/parameters first; then press ▶ or Shift+Enter.
- Outputs and plots will render below; re-run after edits.

In [ ]:
# Results Visualization UI (histograms, CDF, optional KDE, log-scale)

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import re, json as _json
    from pathlib import Path as _Path
    import numpy as _np
    import matplotlib.pyplot as _plt
    from scipy.stats import gaussian_kde as _kde

    results_dir = RESULTS_DIR
    npz_files = sorted(results_dir.glob('*.npz'), key=lambda p: p.stat().st_mtime)
    json_files = sorted(results_dir.glob('*_summary.json'), key=lambda p: p.stat().st_mtime)

    npz_dd = widgets.Dropdown(options=[(p.name, str(p)) for p in npz_files] or [('(none)','')], description='Samples:')
    json_dd = widgets.Dropdown(options=[(p.name, str(p)) for p in json_files] or [('(none)','')], description='Summary:')
    refresh_btn = widgets.Button(description='Refresh list')
    plot_btn = widgets.Button(description='Plot', button_style='success')
    kde_chk = widgets.Checkbox(value=False, description='KDE overlay')
    log_chk = widgets.Checkbox(value=False, description='Log X')
    reconstruct_chk = widgets.Checkbox(value=True, description='Reconstruct R if missing')
    status = widgets.HTML(value='')
    out = widgets.Output()

    def _refresh(_=None):
        # Recompute file lists locally; update dropdowns without nonlocal reassignment
        _npz_files = sorted(results_dir.glob('*.npz'), key=lambda p: p.stat().st_mtime)
        _json_files = sorted(results_dir.glob('*_summary.json'), key=lambda p: p.stat().st_mtime)
        npz_dd.options = [(p.name, str(p)) for p in _npz_files] or [('(none)','')]
        json_dd.options = [(p.name, str(p)) for p in _json_files] or [('(none)','')]
        status.value = "<span style='color:#5cb85c'>Refreshed.</span>"

    def _load_npz(path):
        if not path: return None
        try:
            return _np.load(_Path(path), allow_pickle=True)
        except Exception as e:
            status.value = f"<span style='color:#d9534f'>Failed to load npz: {e}</span>"
            return None

    def _try_reconstruct_R(data):
        if 'R' in data: return _np.asarray(data['R'])
        files = list(data.files)
        m_keys = [k for k in files if k.startswith('m_')]
        name_keys = ['marginal_names','var_names','columns','names','m_names','marginal_keys']
        var_names = None
        for nk in name_keys:
            if nk in files:
                arr = data[nk]
                if isinstance(arr,_np.ndarray) and arr.dtype==object: arr = arr.tolist()
                var_names = list(arr); break
        if m_keys and var_names:
            sampled = {}
            for mk in m_keys:
                key = mk[2:]
                if key in var_names:
                    sampled[key] = _np.asarray(data[mk])
            try:
                metrics = _compute_metrics_cpu(sampled)
                if 'R' in metrics: return _np.asarray(metrics['R'])
            except Exception:
                pass
        return None

    def _plot(_=None):
        with out:
            clear_output(wait=True)
            p = npz_dd.value
            data = _load_npz(p) if p else None
            R = None
            if data is not None:
                if 'R' in data:
                    R = _np.asarray(data['R'])
                elif reconstruct_chk.value:
                    R = _try_reconstruct_R(data)
            if R is None:
                print('No R found; nothing to plot.')
                return
            n = len(R); mean = _np.mean(R); median = _np.median(R)
            fig, axes = _plt.subplots(1,2, figsize=(11,4))
            bins = 120 if n>10000 else max(40, int(n//80))
            axes[0].hist(R, bins=bins, color='#55A868', alpha=0.8)
            if kde_chk.value:
                try:
                    xs = _np.linspace(R.min(), R.max(), 300)
                    kde = _kde(R)
                    axes[0].plot(xs, kde(xs)*(R.max()-R.min())/bins, color='black', lw=1.2)
                except Exception:
                    pass
            axes[0].set_title(f"Histogram R\nmean={mean:.3e} median={median:.3e}")
            if log_chk.value and _np.all(R>0):
                axes[0].set_xscale('log')
            axes[0].grid(True, alpha=0.3)
            axes[1].plot(_np.sort(R), _np.linspace(0,1,n), lw=2, color='#4C72B0')
            axes[1].set_title('Empirical CDF of R')
            axes[1].grid(True, alpha=0.3)
            fig.tight_layout(); _plt.show()

    refresh_btn.on_click(_refresh)
    plot_btn.on_click(_plot)
    panel = widgets.VBox([
        widgets.HBox([npz_dd, json_dd]),
        widgets.HBox([reconstruct_chk, kde_chk, log_chk]),
        widgets.HBox([refresh_btn, plot_btn]),
        status, out
    ])
    display(panel)
except Exception as e:
    print('Results UI unavailable:', e)


# Results visualization: ECDF and KDE (equations)

Empirical CDF of $R$:
$$
\hat F_n(x) = \frac{1}{n}\sum_{i=1}^n \mathbf 1\{R_i \le x\}.
$$

Kernel density estimate (optional overlay) with Gaussian kernel $K$ and bandwidth $h$:
$$
\hat f_h(x) = \frac{1}{n h}\sum_{i=1}^n K\!\left(\frac{x - R_i}{h}\right),\qquad K(t)=\frac{1}{\sqrt{2\pi}}e^{-t^2/2}.
$$

We also provide log-x scaling when $R>0$ and spans orders of magnitude.


### How to use the next cell

- Run the next code cell to execute this visualization or computation block.
- Adjust inputs above if needed; then press ▶ or Shift+Enter.
- Inspect the output below; re-run after changes.

In [16]:
# 14) Feasibility Report UI (R_target thresholding, reconstruction path)
# Simplified: remove datetime usage; rely on local computer clock via time module.
# Filename now uses epoch seconds; report includes 'generated_epoch' and 'generated_local' (localtime).

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import json as _json
    import numpy as _np
    import time as _time

    r_target_txt = widgets.FloatText(value=1e-6, description='R target:')
    regen_btn = widgets.Button(description='Generate report', button_style='success')
    reconstruct_chk2 = widgets.Checkbox(value=True, description='Reconstruct R if needed')
    status2 = widgets.HTML(value='')
    report_out = widgets.Output()

    def _load_R_from_npz(path):
        try:
            data = _np.load(path, allow_pickle=True)
        except Exception as e:
            status2.value = f"<span style='color:#d9534f'>Load failed: {e}</span>"; return None
        if 'R' in data:
            return _np.asarray(data['R'])
        if reconstruct_chk2.value:
            files = list(data.files)
            m_keys = [k for k in files if k.startswith('m_')]
            sampled = {mk[2:]: _np.asarray(data[mk]) for mk in m_keys}
            try:
                metrics = _compute_metrics_cpu(sampled)
                if 'R' in metrics:
                    return _np.asarray(metrics['R'])
            except Exception:
                pass
        return None

    def _generate(_):
        with report_out:
            clear_output(wait=True)
            R_target = float(r_target_txt.value)
            files = sorted(RESULTS_DIR.glob('*.npz'), key=lambda p: p.stat().st_mtime)
            if not files:
                print('No results found.'); status2.value = '<span style="color:#d9534f">No results.</span>'; return
            latest = files[-1]
            R = _load_R_from_npz(latest)
            epoch = int(_time.time())
            local_str = _time.strftime('%Y-%m-%d %H:%M:%S')
            report = {'generated_epoch': epoch, 'generated_local': local_str, 'R_target': R_target, 'source': latest.name}
            if R is None:
                report['R_present'] = False
                report['verdict'] = 'No R metric available.'
            else:
                R = _np.asarray(R)
                report['R_present'] = True
                report['n_samples'] = int(len(R))
                report['mean_R'] = float(_np.mean(R))
                report['median_R'] = float(_np.median(R))
                p10, p90 = _np.percentile(R, [10, 90])
                report['p10_R'] = float(p10); report['p90_R'] = float(p90)
                frac_above = float((R >= R_target).mean())
                report['frac_above_R_target'] = frac_above
                if frac_above >= 0.5:
                    verdict = 'Likely feasible'
                elif frac_above > 0.0:
                    verdict = 'Partially feasible'
                else:
                    verdict = 'Likely not feasible'
                report['verdict'] = verdict
            print(_json.dumps(report, indent=2))
            fname = RESULTS_DIR / (f'{epoch}_feasibility_report.json')
            with open(fname, 'w', encoding='utf-8') as f:
                _json.dump(report, f, indent=2)
            status2.value = f"<span style='color:#5cb85c'>Report saved to {fname.name}</span>"

    regen_btn.on_click(_generate)
    panel2 = widgets.VBox([
        widgets.HTML('<b>Feasibility report</b>'),
        widgets.HBox([r_target_txt, reconstruct_chk2, regen_btn]),
        status2,
        report_out
    ])
    display(panel2)
except Exception as e:
    print('Feasibility UI unavailable:', e)

# Feasibility report generator: statistical interpretation

Reported fraction above target:
$$
\hat p = \frac{1}{n}\sum_{i=1}^n \mathbf 1\{R_i \ge R_{\text{target}}\}.
$$
For large $n$, a descriptive Wald interval could be formed $\hat p \pm z_{\alpha/2}\sqrt{\hat p(1-\hat p)/n}$ (Wald 1941; caution for extreme $\hat p$). More stable alternatives include Wilson (Wilson 1927 https://doi.org/10.2307/2280061) or exact Clopper–Pearson (Clopper & Pearson 1934 https://doi.org/10.1093/biomet/26.4.404). Verdict thresholds (≥50%, ≥10%, <10%) here are heuristic feasibility bands, not formal hypothesis tests.


### How to use the next cell

- Execute the next code cell to generate the combined report/summary.
- Ensure prior sampling and metrics cells have been run; then press ▶ or Shift+Enter.
- The output will show the report and any saved file paths.

In [17]:
# 15) Detailed Feasibility Report (Plotly histogram + ECDF, executive summary)
# Idempotent rendering avoiding duplicates; toggle best sample; JSON export; sensitivity proxy.

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML
    import json as _json
    import numpy as _np
    try:
        import plotly.graph_objects as go
        _HAS_PLOTLY_DET = True
    except Exception:
        _HAS_PLOTLY_DET = False

    # Cleanup prior detailed report widgets
    _DRW = globals().get('_detailed_report_widgets')
    if _DRW:
        for w in _DRW:
            try: w.close()
            except Exception: pass
    _DRW = []

    def _register(*ws):
        for w in ws: _DRW.append(w)
        globals()['_detailed_report_widgets'] = _DRW

    def _extract_R(report_dict):
        for k in ('R','R_array','R_samples'):
            if isinstance(report_dict.get(k), (list,_np.ndarray)):
                arr = _np.asarray(report_dict[k], dtype=float)
                if arr.size: return arr
        # Fallback look for metrics
        m = report_dict.get('metrics')
        if isinstance(m, dict) and 'R' in m:
            arr = _np.asarray(m['R'], dtype=float)
            if arr.size: return arr
        return None

    def display_detailed_report(report=None, R_target=None):
        if report is None:
            # Attempt to find latest feasibility report
            files = sorted(RESULTS_DIR.glob('*_feasibility_report.json'), key=lambda p: p.stat().st_mtime)
            if files:
                try:
                    report = _json.load(open(files[-1],'r',encoding='utf-8'))
                except Exception:
                    report = None
        if report is None:
            display(HTML('<b>No report found.</b>'))
            return
        R_arr = None
        if report.get('R_present') and 'frac_above_R_target' in report and 'mean_R' in report:
            # Only summary stored, try reconstruction via latest samples
            files = sorted(RESULTS_DIR.glob('*.npz'), key=lambda p: p.stat().st_mtime)
            if files:
                data = _np.load(files[-1], allow_pickle=True)
                if 'R' in data:
                    R_arr = _np.asarray(data['R'])
        if R_arr is None:
            R_arr = _extract_R(report)
        if R_arr is None or R_arr.size == 0 or not _np.isfinite(R_arr).any():
            display(HTML('<b>Unable to obtain R array for detailed report.</b>'))
            return
        R_arr = _np.nan_to_num(R_arr, nan=0.0, posinf=0.0, neginf=0.0)
        if R_target is None:
            R_target = report.get('R_target', 1e-6)
        R_target = float(R_target)

        mean_R = float(_np.mean(R_arr)); median_R = float(_np.median(R_arr))
        p10, p90 = _np.percentile(R_arr, [10, 90])
        frac_above = float(_np.mean(R_arr >= R_target))
        best_idx = int(_np.argmax(R_arr)); best_R = float(R_arr[best_idx])
        if frac_above >= 0.5:
            verdict, color = 'Feasible', '#2ca02c'
        elif frac_above >= 0.1:
            verdict, color = 'Marginal', '#ff7f0e'
        else:
            verdict, color = 'Not feasible', '#d62728'

        header_html = (
            "<div style='border:1px solid #ddd;padding:10px;border-radius:6px;background:#fafafa'>"
            f"<h3 style='margin:0'>Detailed Feasibility Report</h3>"
            f"<div style='margin-top:6px'><span style='padding:6px 10px;background:{color};color:#fff;border-radius:4px;font-weight:600'>{verdict}</span>"
            f" <span style='margin-left:12px;color:#333;'>R_target = <b>{R_target:.3g}</b></span></div></div>"
        )

        metrics_html = "<table style='width:100%;border-collapse:collapse;margin-top:8px'>" \
            + f"<tr><td><b>Mean R</b></td><td>{mean_R:.4g}</td><td><b>Median R</b></td><td>{median_R:.4g}</td></tr>" \
            + f"<tr><td><b>10th %</b></td><td>{p10:.4g}</td><td><b>90th %</b></td><td>{p90:.4g}</td></tr>" \
            + f"<tr><td><b>Frac ≥ target</b></td><td>{frac_above:.3%}</td><td><b>Best R</b></td><td>{best_R:.4g}</td></tr>" \
            + "</table>"

        plot_out = widgets.Output()
        with plot_out:
            if _HAS_PLOTLY_DET:
                # Build Plotly figures
                n = int(R_arr.size)
                nbins = max(30, min(200, n // 80 if n > 0 else 30))
                base_h = go.Figure()
                base_h.add_trace(go.Histogram(x=R_arr, nbinsx=nbins, marker_color='#1f77b4', opacity=0.85))
                base_h.add_vline(x=R_target, line=dict(color='red', dash='dash'))
                base_h.update_layout(width=520, height=280, title='R histogram')
                base_c = go.Figure()
                sorted_R = _np.sort(R_arr)
                cdf = _np.arange(1,len(sorted_R)+1)/float(len(sorted_R))
                base_c.add_trace(go.Scatter(x=sorted_R, y=cdf, mode='lines', line=dict(color='#2ca02c')))
                base_c.add_vline(x=R_target, line=dict(color='red', dash='dash'))
                base_c.update_layout(width=520, height=280, title='R ECDF')
                # Use FigureWidget to render reliably in VS Code/Notebook without injecting HTML
                try:
                    fw_h = go.FigureWidget(base_h)
                    fw_c = go.FigureWidget(base_c)
                    display(widgets.HBox([
                        widgets.VBox([widgets.HTML('<b>Distribution</b>'), fw_h]),
                        widgets.VBox([widgets.HTML('<b>ECDF</b>'), fw_c])
                    ]))
                except Exception:
                    # Fallback to direct display if FigureWidget not available
                    display(base_h)
                    display(base_c)
            else:
                print('Plotly not available; install plotly for interactive figures.')

        best_toggle = widgets.ToggleButton(value=False, description='Show best sample')
        best_out = widgets.Output()
        save_btn = widgets.Button(description='Export JSON')
        save_msg = widgets.HTML()

        # Sensitivity proxy: rank correlation (Spearman approx via rank correlation formula)
        sens_out = widgets.Output()
        with sens_out:
            lines = ["<div style='font-size:90%;color:#333'>"]
            # Attempt to reconstruct sample dict if available
            # This is heuristic: look for most recent npz
            npz_files = sorted(RESULTS_DIR.glob('*.npz'), key=lambda p: p.stat().st_mtime)
            if npz_files:
                try:
                    data = _np.load(npz_files[-1], allow_pickle=True)
                    sample_vars = [k for k in data.files if k.startswith('m_')]
                    for sv in sample_vars:
                        arr = _np.asarray(data[sv])
                        if arr.size == R_arr.size:
                            try:
                                r_corr = _np.corrcoef(_np.argsort(arr), _np.argsort(R_arr))[0,1]
                                if _np.isfinite(r_corr):
                                    lines.append(f"<div><b>{sv[2:]}</b>: sensitivity proxy = {r_corr:.3f}</div>")
                            except Exception:
                                pass
                except Exception:
                    pass
            lines.append("</div>")
            display(HTML("".join(lines)))

        def _on_best(change):
            if change['name']=='value':
                best_out.clear_output()
                if change['new']:
                    with best_out:
                        idx = best_idx
                        display(HTML(f"<div style='border:1px solid #eee;padding:8px;border-radius:6px;background:#fff'>Best sample index: {idx}<br/>R={best_R:.4g}</div>"))
        best_toggle.observe(_on_best, names='value')

        def _on_save(_):
            fname = RESULTS_DIR / 'feasibility_report_export.json'
            try:
                with open(fname,'w',encoding='utf-8') as fh:
                    _json.dump(report, fh, indent=2)
                save_msg.value = f"<span style='color:green'>Saved: {fname.name}</span>"
            except Exception as e:
                save_msg.value = f"<span style='color:red'>Save failed: {e}</span>"
        save_btn.on_click(_on_save)

        exec_summary = [
            f"Target <b>{R_target:.3g}</b> Hz; fraction ≥ target <b>{frac_above:.1%}</b>; verdict <b>{verdict}</b>.",
            f"Median R = {median_R:.3g}; mean R = {mean_R:.3g}; (P10,P90)=({p10:.3g},{p90:.3g}); best R = {best_R:.3g}."
        ]
        if best_R < R_target:
            exec_summary.append("Best sample below target — improve high-influence parameters (efficiencies, losses, multiplexing).")
        else:
            exec_summary.append("At least one configuration reaches target — explore neighborhood for robustness.")
        exec_html = "<div style='margin-top:10px;padding:8px;border-left:4px solid #888;background:#f9f9f9'>" + " ".join(exec_summary) + "</div>"

        controls = widgets.HBox([best_toggle, save_btn, save_msg])
        left_col = widgets.VBox([widgets.HTML(header_html), widgets.HTML(metrics_html), controls, best_out, widgets.HTML(exec_html), sens_out])
        det_panel = widgets.HBox([left_col, plot_out])
        display(det_panel)
        _register(best_toggle, save_btn, save_msg, plot_out, left_col, det_panel, sens_out, best_out)

    # UI container with R_target control
    r_target_det = widgets.FloatText(value=1e-6, description='R target (detail)')
    render_btn = widgets.Button(description='Render detailed', button_style='info')
    out_det = widgets.Output()

    def _on_render(_):
        out_det.clear_output()
        with out_det:
            display_detailed_report(None, R_target=r_target_det.value)
    render_btn.on_click(_on_render)

    det_container = widgets.VBox([
        widgets.HTML('<b>Detailed feasibility report</b>'),
        widgets.HBox([r_target_det, render_btn]),
        out_det
    ])
    display(det_container)
    _register(det_container, r_target_det, render_btn, out_det)
except Exception as e:
    print('Detailed report UI unavailable:', e)

# Detailed report: sensitivity proxy (rank-based)

We approximate monotonic influence of each marginal \(m_k\) on \(R\) via rank correlation. Let \(r(m_k), r(R)\) be rank vectors. The Spearman correlation can be computed as Pearson on ranks, or via
\[
\rho_s = 1 - \frac{6\sum_{i=1}^n d_i^2}{n(n^2-1)},\quad d_i = r(m_k)_i - r(R)_i,
\]
when there are no ties (otherwise use Pearson on ranks). High \(|\rho_s|\) indicates stronger monotonic association under the sampled regime.


### How to use the next cell

- Run the next code cell to perform this final computation or export.
- Confirm all upstream cells have been executed; then press ▶ or Shift+Enter.
- Check the output for saved artifact paths and summary statistics.

In [18]:
# Test config & copula for end-to-end run (safe defaults; overrides CONFIG)
try:
    # Resolve a seed even if the constants cell hasn't run yet
    _seed = int(globals().get('GLOBAL_SEED', 123456789))
    # Minimal CONFIG overlay; SCHEMA provides other defaults
    CONFIG = {
        'preset_id': 'test_preset',
        'alpha_dB_per_km': 0.2,
        'L0_km': 50.0,
        'eta_s': 0.7,
        'eta_w': 0.6,
        'eta_r': 0.6,
        'eta_det': 0.8,
        'V0': 0.9,
        'T2_ms': 150.0,
        'N_modes': 100,
        'n_swaps': 3,
        'kappa_power': 1.0,
        'c_fiber_mps': 2.0e8,
        'connector_loss_dB': 0.5,
        'n_connectors': 10,
        'splice_loss_dB': 0.1,
        'n_splices': 200,
        'filter_loss_dB': 1.0,
        'p_dark': 0.0,
        'raman_rate_per_ns': 0.0,
        'gate_ns': 1.0,
        'geometry': 'central_bsm',
        'kappa_mode': 'power',
        'mc': {'samples': 5000, 'seed': _seed},
        'copula': {'kendall_tau_pairs': [
            ('eta_s','eta_det', 0.25),
            ('eta_s','V0', 0.15),
            ('V0','T2_ms', -0.20)
        ]}
    }
    # Define a clean WORKING_MARGINALS for this test (stochastic subset only)
    # Remove c_fiber_mps from marginals (treat as deterministic) to avoid PIT flake; physics reads it from CONFIG.
    import math as _math
    WORKING_MARGINALS = {
        'eta_s': {'family':'beta','params':{'a':70,'b':30},'unit':'frac'},
        'eta_det': {'family':'beta','params':{'a':75,'b':25},'unit':'frac'},
        'alpha_dB_per_km': {'family':'lognormal','params':{'mu_log':float(_math.log(CONFIG['alpha_dB_per_km'])), 'sigma_log':0.05},'unit':'dB/km'},
        'L0_km': {'family':'uniform','params':{'low':CONFIG['L0_km']*0.8,'high':CONFIG['L0_km']*1.2},'unit':'km'},
        'V0': {'family':'beta','params':{'a':90,'b':10},'unit':'vis'},
        'T2_ms': {'family':'lognormal','params':{'mu_log':float(_math.log(max(CONFIG['T2_ms'],1e-9))),'sigma_log':0.1},'unit':'ms'},
        'N_modes': {'family':'uniform','params':{'low':CONFIG['N_modes']*0.9,'high':CONFIG['N_modes']*1.1},'unit':'count'},
        'n_swaps': {'family':'uniform','params':{'low':max(CONFIG.get('n_swaps',0)-1,0),'high':CONFIG.get('n_swaps',0)+1},'unit':'count'},
        'kappa_power': {'family':'uniform','params':{'low':1.0,'high':1.2},'unit':'power'}
    }
    # Clear prior state for a clean run
    FLOW_STATE = {}
    print('[Test] CONFIG & WORKING_MARGINALS prepared; FLOW_STATE reset.')
except Exception as e:
    print('Test config setup failed:', e)

[Test] CONFIG & WORKING_MARGINALS prepared; FLOW_STATE reset.
